네이버 통합 모델

타겟: 지그재그(당일/익일), 카카오, 공구우먼, 네이버_통합, 순수_기타물량

특징: shop_id 134, 141, 143, 144번의 물량을 모두 더해 네이버_통합으로 예측

## Total Forecasting 

### 초반 설정 (패키지 설치)

In [42]:
%pip install pandas numpy holidays scikit-learn xgboost pymysql sshtunnel python-dotenv openpyxl requests tqdm urllib3 ipykernel 

Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [43]:
!pip3 uninstall -y paramiko
!pip3 install paramiko==3.5.1 sshtunnel==0.4.0

Found existing installation: paramiko 3.5.1
Uninstalling paramiko-3.5.1:
  Successfully uninstalled paramiko-3.5.1
Defaulting to user installation because normal site-packages is not writeable
  Using cached paramiko-3.5.1-py3-none-any.whl (227 kB)
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.


In [44]:
import sys
print(sys.executable)

/Library/Developer/CommandLineTools/usr/bin/python3


In [45]:
import pandas as pd
import pymysql
from sshtunnel import SSHTunnelForwarder
from dotenv import load_dotenv
import requests
import numpy as np
from tqdm import tqdm

print("import 성공")

import 성공


In [46]:
pip install "paramiko<4"

Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


이전 데이터에서 부스터스 제외하는 일회성 코드

In [47]:
import os
import pandas as pd
import shutil

# 1. test_data 폴더 생성
os.makedirs("./test_data", exist_ok=True)

print("⏳ 마이그레이션 시작...")

# 2. total_latest.xlsx 변환
if os.path.exists("./data/total_latest.xlsx"):
    df_total = pd.read_excel("./data/total_latest.xlsx")
    
    if '부스터스' in df_total.columns:
        # 부스터스 물량을 총 물량에서 제외 (과거 유령 물량 제거)
        df_total['총 물량'] = df_total['총 물량'] - df_total['부스터스']
        
        # 부스터스 컬럼을 공구우먼으로 변경 후 0으로 처리 (과거엔 공구우먼이 없었으므로)
        df_total.rename(columns={'부스터스': '공구우먼'}, inplace=True)
        df_total['공구우먼'] = 0
        
    df_total.to_excel("./test_data/total_latest.xlsx", index=False)
    print("✅ total_latest.xlsx 변환 완료")

# 3. region_latest_cl.xlsx 변환 (첨부해주신 이미지 참고)
if os.path.exists("./data/region_latest_cl.xlsx"):
    df_region = pd.read_excel("./data/region_latest_cl.xlsx")
    
    if '부스터스' in df_region.columns:
        # 부스터스 물량을 총 물량에서 제외
        df_region['총 물량'] = df_region['총 물량'] - df_region['부스터스']
        
        # 부스터스 컬럼을 공구우먼으로 변경 후 0으로 처리
        df_region.rename(columns={'부스터스': '공구우먼'}, inplace=True)
        df_region['공구우먼'] = 0
        
    df_region.to_excel("./test_data/region_latest_cl.xlsx", index=False)
    print("✅ region_latest_cl.xlsx 변환 완료")

# 4. recent_temp.xlsx 복사 (기온 데이터는 그대로 사용)
if os.path.exists("./data/recent_temp.xlsx"):
    shutil.copy("./data/recent_temp.xlsx", "./test_data/recent_temp.xlsx")
    print("✅ recent_temp.xlsx 복사 완료")

print("🎉 모든 데이터가 test_data 폴더에 성공적으로 구축되었습니다!")

⏳ 마이그레이션 시작...
🎉 모든 데이터가 test_data 폴더에 성공적으로 구축되었습니다!


### Data load & Update

In [38]:
# ==========================================================
# [v.6.4.1] Cell 1. 데이터 로드 + 기온 병합 + 6-Track 타겟 생성 (네이버 통합)
# ==========================================================

import os
import numpy as np
import pandas as pd
import pymysql
import requests
import urllib3

from dotenv import load_dotenv
from sshtunnel import SSHTunnelForwarder
from tqdm import tqdm
from datetime import datetime, timedelta

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
load_dotenv()

# 💡 해당 버전의 전용 폴더 경로 설정
VERSION_DIR = "./v.6.4.1"
DATA_DIR = f"{VERSION_DIR}/test_data"
os.makedirs(DATA_DIR, exist_ok=True)

# ==========================================================
# 1. DB 데이터 로드 (네이버 통합 추출)
# ==========================================================
class AutoContainerGeneration_v6_4_1:
    def __init__(self, version="6.4.1", debug=False):
        self.version = version
        self.debug = debug
        print(f"AutoContainerGeneration version: {version}")

    def fetch_data(self, query, ssh_host, ssh_user, ssh_private_key, mysql_host, mysql_port, mysql_user, mysql_password, mysql_database):
        try:
            with SSHTunnelForwarder(
                (ssh_host, 22),
                ssh_username=ssh_user,
                ssh_private_key=ssh_private_key,
                remote_bind_address=(mysql_host, mysql_port),
            ) as tunnel:
                print("SSH 터널 연결 성공")
                with pymysql.connect(
                    host="127.0.0.1", user=mysql_user, passwd=mysql_password, db=mysql_database,
                    charset="utf8", port=tunnel.local_bind_port, cursorclass=pymysql.cursors.DictCursor
                ) as conn:
                    with conn.cursor() as cur:
                        cur.execute(query)
                        results = cur.fetchall()
                        print("쿼리 실행 완료")
                        return pd.DataFrame(results)
        except Exception as e:
            print(f"Error fetching data: {e}")
            return None

    def fetch_cluster_data(self):
        ssh_host = os.getenv("SSH_HOST_VER_1")
        ssh_user = os.getenv("SSH_USER")
        ssh_private_key = os.getenv("SSH_PRIVATE_KEY")
        mysql_host = os.getenv("MYSQL_HOST")
        mysql_port = 3306
        mysql_user = os.getenv("MYSQL_USER")
        mysql_password = os.getenv("MYSQL_PASSWORD")
        mysql_database = os.getenv("MYSQL_DATABASE")


        cluster_query = """
            SELECT
                p.plan_date AS 날짜,
                COUNT(DISTINCT si.id) AS "총 물량",
                SUM(CASE WHEN s.name = '지그재그 당일배송' THEN CASE WHEN JSON_UNQUOTE(JSON_EXTRACT(o.metadata, '$.order_items[0].product.delivery_type')) IS NULL AND HOUR(o.`timestamp`) < 13 THEN 1 ELSE 0 END ELSE 0 END) AS "지그재그_당일",
                SUM(CASE WHEN s.name = '지그재그 당일배송' THEN CASE WHEN JSON_UNQUOTE(JSON_EXTRACT(o.metadata, '$.order_items[0].product.delivery_type')) IS NULL AND HOUR(o.`timestamp`) >= 13 THEN 1 ELSE 0 END ELSE 0 END) AS "지그재그_익일",
                SUM(CASE WHEN s.name = '카카오모빌리티' THEN 1 ELSE 0 END) AS "카카오모빌리티",
                SUM(CASE WHEN s.name = '공구우먼' THEN 1 ELSE 0 END) AS "공구우먼",
                SUM(CASE WHEN s.id IN (134, 141, 143, 144) THEN 1 ELSE 0 END) AS "네이버_통합",
                SUM(CASE WHEN s.name = '지그재그 당일배송' THEN 1 ELSE 0 END) AS "지그재그 당일배송",
                COUNT(DISTINCT s.name) AS "셀러수"
            FROM (
                SELECT id, plan_date 
                FROM hub_cluster_plan 
                WHERE plan_date >= '2026-06-01' AND plan_date <= CURRENT_DATE()
                AND ((plan_date < '2026-07-02' AND hub_cluster_plan.plan_no IN (1, 90)) OR (plan_date >= '2026-07-02' AND hub_cluster_plan.plan_no = 10))
            ) p
            JOIN hub_cluster_plan_item pi ON p.id = pi.cluster_plan_id
            JOIN shipping_shippingitem si ON pi.shipping_item_id = si.id
            JOIN order_order o ON si.order_id = o.id
            JOIN location_sector ls ON si.designated_sector_id = ls.id
            JOIN shop_shop s ON o.shop_id = s.id
            WHERE ls.allowed = 1
            AND s.name != '부스터스'
            AND s.name NOT LIKE '%테스트%' 
            GROUP BY p.plan_date
            ORDER BY p.plan_date;
        """
        return self.fetch_data(cluster_query, ssh_host, ssh_user, ssh_private_key, mysql_host, mysql_port, mysql_user, mysql_password, mysql_database)

# ==========================================================
# 2. 마스터 데이터 병합
# ==========================================================
generator = AutoContainerGeneration_v6_4_1()
df_forecast = generator.fetch_cluster_data()
file_path = f"{DATA_DIR}/total_latest.xlsx"

if df_forecast is None or df_forecast.empty:
    print(f"🚨 DB에서 데이터를 가져오지 못했습니다. 기존 {file_path}만 로드합니다.")
    data = pd.read_excel(file_path)
else:
    print("✅ 최신 DB 데이터 추출 완료")
    db_df = df_forecast.copy()
    db_df.columns = [c.replace("총_물량", "총 물량") for c in db_df.columns]
    db_df["날짜"] = pd.to_datetime(db_df["날짜"], errors="coerce").dt.normalize()

    try:
        base = pd.read_excel(file_path)
        base["날짜"] = pd.to_datetime(base["날짜"], errors="coerce").dt.normalize()
    except FileNotFoundError:
        base = pd.DataFrame()

    if not base.empty and not db_df.empty:
        base = base[~base["날짜"].isin(db_df["날짜"])]

    data = pd.concat([base, db_df], axis=0).reset_index(drop=True)
    data = data.sort_values("날짜").reset_index(drop=True)

# ==========================================================
# 3. 6-Track 타겟 생성 (네이버 통합 반영)
# ==========================================================
required_cols = ["날짜", "총 물량", "지그재그_당일", "지그재그_익일", "카카오모빌리티", "공구우먼", "네이버_통합", "지그재그 당일배송", "셀러수"]
for col in required_cols:
    if col not in data.columns: data[col] = 0

data["날짜"] = pd.to_datetime(data["날짜"], errors="coerce").dt.normalize()

numeric_cols = [c for c in required_cols if c != "날짜"]
for col in numeric_cols:
    data[col] = pd.to_numeric(data[col], errors="coerce").fillna(0)

# 순수 기타물량 계산
data["순수_기타물량"] = (data["총 물량"] - data["지그재그_당일"] - data["지그재그_익일"] - data["카카오모빌리티"] - data["공구우먼"] - data["네이버_통합"]).clip(lower=0)

final_cols = ["날짜", "총 물량", "지그재그_당일", "지그재그_익일", "카카오모빌리티", "공구우먼", "네이버_통합", "순수_기타물량", "지그재그 당일배송", "셀러수"]
data = data[final_cols].sort_values("날짜").reset_index(drop=True)
data.to_excel(file_path, index=False)
print(f"✨ 6-Track 마스터 데이터 저장 완료 ({file_path})")


# ==========================================================
# 4. 기온 데이터 로드/갱신/병합 (폴더 경로 수정)
# ==========================================================
serviceKey = os.getenv("KMA_API_KEY")

def fetch_short_term_forecast(base_date: str, base_time: str, region_point: dict):
    records_all = []
    if serviceKey is None: return pd.DataFrame(columns=["날짜", "지역", "avg_temp"])
    for region, (nx, ny) in tqdm(region_point.items(), desc="단기예보 수집"):
        url = f"https://apihub.kma.go.kr/api/typ02/openApi/VilageFcstInfoService_2.0/getVilageFcst?pageNo=1&numOfRows=10000&dataType=JSON&base_date={base_date}&base_time={base_time}&nx={nx}&ny={ny}&authKey={serviceKey}"
        try:
            response = requests.get(url, verify=False, timeout=20)
            items = response.json()["response"]["body"]["items"]["item"]
        except Exception: continue

        informations = {}
        for it in items:
            key = f"{it['fcstDate']}_{it['fcstTime']}"
            if key not in informations: informations[key] = {}
            informations[key][it['category']] = it['fcstValue']

        region_records = [{"date": k.split("_")[0], "TMP": float(v["TMP"])} for k, v in informations.items() if "TMP" in v]
        if not region_records: continue
        df_region = pd.DataFrame(region_records).groupby("date", as_index=False)["TMP"].mean().rename(columns={"TMP": "avg_temp"})
        df_region["지역"] = region
        records_all.append(df_region)
    if not records_all: return pd.DataFrame(columns=["날짜", "지역", "avg_temp"])
    df_all = pd.concat(records_all, ignore_index=True).rename(columns={"date": "날짜"})
    df_all["날짜"] = pd.to_datetime(df_all["날짜"], format="%Y%m%d").dt.strftime("%Y-%m-%d")
    return df_all

def fetch_mid_term_forecast(base_date: str, region_code: dict):
    records = []
    if serviceKey is None: return pd.DataFrame(columns=["날짜", "지역", "avg_temp"])
    for region, regId in tqdm(region_code.items(), desc="중기예보 수집"):
        url = f"https://apihub.kma.go.kr/api/typ02/openApi/MidFcstInfoService/getMidTa?pageNo=1&numOfRows=10000&dataType=JSON&regId={regId}&tmFc={base_date}0600&authKey={serviceKey}"
        try:
            response = requests.get(url, timeout=20)
            item = response.json()["response"]["body"]["items"]["item"][0]
        except Exception: continue
        for day in range(4, 11):
            ta_min, ta_max = item.get(f"taMin{day}"), item.get(f"taMax{day}")
            if ta_min is not None and ta_max is not None:
                records.append({"날짜": (datetime.now() + timedelta(days=day)).strftime("%Y-%m-%d"), "지역": region, "avg_temp": (ta_min + ta_max) / 2})
    return pd.DataFrame(records, columns=["날짜", "지역", "avg_temp"])

region_point = {"서울특별시": ("61", "126"), "인천광역시": ("54", "125"), "경기도": ("60", "121"), "대전광역시": ("68", "100"), "충청남도": ("65", "110")}
region_code = {"서울특별시": "11B10101", "인천광역시": "11B20201", "경기도": "11B20601", "대전광역시": "11C20401", "충청남도": "11C20101"}

short_df = fetch_short_term_forecast(datetime.now().strftime("%Y%m%d"), "0200", region_point)
mid_df = fetch_mid_term_forecast(datetime.now().strftime("%Y%m%d"), region_code)

combined_df = pd.concat([short_df, mid_df], ignore_index=True)
if not combined_df.empty:
    combined_df["날짜"] = pd.to_datetime(combined_df["날짜"]).dt.strftime("%Y-%m-%d")
    final_df = combined_df.drop_duplicates(subset=["날짜", "지역"], keep="first").sort_values(["지역", "날짜"]).reset_index(drop=True)
    final_df["avg_temp"] = np.round(final_df["avg_temp"], 1)
    final_df = final_df[["날짜", "지역", "avg_temp"]]
else:
    final_df = pd.DataFrame(columns=["날짜", "지역", "avg_temp"])

temp_file_path = f"{DATA_DIR}/recent_temp.xlsx"
try:
    base_temp = pd.read_excel(temp_file_path)
    base_temp["날짜"] = pd.to_datetime(base_temp["날짜"], errors="coerce")
except FileNotFoundError:
    base_temp = pd.DataFrame(columns=["날짜", "지역", "avg_temp"])

temp_data = pd.concat([base_temp, final_df], axis=0).reset_index(drop=True)
temp_data["날짜"] = pd.to_datetime(temp_data["날짜"], errors="coerce")
temp_data = temp_data.drop_duplicates(subset=["날짜", "지역"], keep="last").sort_values(["지역", "날짜"]).reset_index(drop=True)
temp_data.to_excel(temp_file_path, index=False)

temp_merge_data = temp_data.groupby("날짜", as_index=False)["avg_temp"].mean()
temp_merge_data["avg_temp"] = np.round(temp_merge_data["avg_temp"], 1)
temp_merge_data["날짜"] = pd.to_datetime(temp_merge_data["날짜"], errors="coerce").dt.normalize()

data = pd.merge(data, temp_merge_data, how="left", on="날짜")
print("✅ 기온 데이터 병합 완료")
display(data.tail(7))

AutoContainerGeneration version: 6.4.1
SSH 터널 연결 성공
쿼리 실행 완료
✅ 최신 DB 데이터 추출 완료
✨ 6-Track 마스터 데이터 저장 완료 (./v.6.4.1/test_data/total_latest.xlsx)
✅ 기온 데이터 병합 완료


,날짜,총 물량,지그재그_당일,지그재그_익일,카카오모빌리티,공구우먼,네이버_통합,순수_기타물량,지그재그 당일배송,셀러수,avg_temp
1021,2026-10-21,26778,5005,6230,798,316,10735,3694,11235,30,NaN
1022,2026-10-22,28956,6339,7152,883,338,10578,3666,13491,30,NaN
1023,2026-10-23,26872,6034,6682,885,409,10037,2825,12716,30,NaN
1024,2026-10-24,17507,3780,4151,309,388,7792,1087,7931,19,NaN
1025,2026-10-25,21282,4146,3733,208,1,10768,2426,7879,19,NaN
1026,2026-10-26,31201,5104,4986,1003,446,15909,3753,10090,31,NaN
1027,2026-10-27,30389,4466,5647,932,757,15501,3086,10113,31,NaN


### Feature & Modeling

In [39]:
# ==========================================================
# Cell 2. Feature + Model 함수 전체 (네이버 통합 추가)
# ==========================================================

import numpy as np
import pandas as pd
import holidays

from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_percentage_error


# ==========================================================
# 0. 5-Track 타겟 정의
# ==========================================================
ZIG_SAME = "지그재그_당일"
ZIG_NEXT = "지그재그_익일"
KAKAO = "카카오모빌리티"
GONGGU = "공구우먼"
NAVER_COMBINED = "네이버_통합"
OTHER_PURE = "순수_기타물량"
TOTAL = "총 물량"

BASE_TARGETS = [
    ZIG_SAME,
    ZIG_NEXT,
    KAKAO,
    GONGGU,
    NAVER_COMBINED,
    OTHER_PURE,
]

ALL_TARGETS = BASE_TARGETS + [TOTAL]


# ==========================================================
# 1. 공통 Feature 함수
# ==========================================================
def add_temp_shift_features(df, date_col="날짜", temp_col="avg_temp"):
    df = df.copy().sort_values(date_col)

    # ✅ avg_temp를 반드시 숫자로 변환
    df[temp_col] = pd.to_numeric(df[temp_col], errors="coerce")

    # 결측 보정
    df[temp_col] = (
        df[temp_col]
        .ffill()
        .bfill()
        .fillna(15.0)
    )

    temp_ma7_min = df[temp_col].rolling(7, min_periods=1).min()
    diff = temp_ma7_min - df[temp_col]

    df["temp_shift"] = diff.abs().astype(float)

    # ✅ np.sign 대신 안전한 방식
    df["temp_shift_sign"] = 0
    df.loc[diff > 0, "temp_shift_sign"] = 1
    df.loc[diff < 0, "temp_shift_sign"] = -1

    df[["temp_shift", "temp_shift_sign"]] = (
        df[["temp_shift", "temp_shift_sign"]]
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0.0)
    )

    return df


def add_target_lag_rolling_features(
    df,
    target_col,
    lags=(1, 7, 14, 21, 28),
    roll_windows=(7, 14, 21, 28),
    impute_mask=None  # 🌟 보간 마스크
):
    df = df.copy().sort_values("날짜").reset_index(drop=True)

    if target_col not in df.columns:
        for lag in lags:
            df[f"{target_col}_lag_{lag}"] = np.nan
        for w in roll_windows:
            df[f"{target_col}_roll_mean_{w}"] = np.nan
            df[f"{target_col}_roll_std_{w}"] = np.nan
        return df

    s = pd.to_numeric(df[target_col], errors="coerce").copy()

    # 수정: 앞뒤 날짜 평균이 아닌, '7일 전 동일 요일' 값으로 우선 채움 (요일 패턴 유지)
    if impute_mask is not None:
        s.loc[impute_mask] = np.nan
        s = s.fillna(s.shift(7)) # 1차: 저번 주 같은 요일 값으로 보간
        s = s.interpolate(method="linear").bfill().ffill().fillna(0) # 2차: 7일 전도 결측일 경우를 대비한 선형 보간 백업

    for lag in lags:
        df[f"{target_col}_lag_{lag}"] = s.shift(lag)

    shifted = s.shift(1)
    for w in roll_windows:
        df[f"{target_col}_roll_mean_{w}"] = shifted.rolling(w, min_periods=max(2, w // 4)).mean()
        df[f"{target_col}_roll_std_{w}"] = shifted.rolling(w, min_periods=max(2, w // 4)).std()

    return df


def build_holiday_set(years, manual_holidays=None):
    """
    holidays.KR 자동 공휴일 + 수동 공휴일을 하나의 set(date)로 통합.
    근로자의 날(5/1), 임시공휴일 등은 manual_holidays로 추가.
    """
    kr_holiday_set = set(holidays.KR(years=years).keys())

    if manual_holidays:
        manual_set = set(pd.to_datetime(manual_holidays).date)
        kr_holiday_set.update(manual_set)

    return kr_holiday_set


def add_pre_post_holiday_features(df, holiday_set):
    """
    공휴일 전후 효과를 생성.
    - pre_holiday_peak: 다음 1~2영업일 내 공휴일이 있으면 피크 가능성
    - post_holiday_peak: 직전 공휴일 이후 첫 영업일
    - days_until_holiday: 다음 공휴일까지 남은 일수
    - days_after_holiday: 직전 공휴일로부터 지난 일수
    """
    df = df.copy()
    df_dates = df["날짜"].dt.date.tolist()
    holiday_dates = sorted(list(holiday_set))

    holiday_ts = pd.to_datetime(holiday_dates)

    df["pre_holiday_peak"] = 0
    df["post_holiday_peak"] = 0
    df["days_until_holiday"] = 99
    df["days_after_holiday"] = 99

    if len(holiday_ts) == 0:
        return df

    date_series = df["날짜"]

    for idx, cur_date in enumerate(date_series):
        cur_norm = pd.Timestamp(cur_date).normalize()

        future_holidays = holiday_ts[holiday_ts >= cur_norm]
        past_holidays = holiday_ts[holiday_ts <= cur_norm]

        if len(future_holidays) > 0:
            days_until = int((future_holidays.min() - cur_norm).days)
            df.loc[idx, "days_until_holiday"] = min(days_until, 99)

        if len(past_holidays) > 0:
            days_after = int((cur_norm - past_holidays.max()).days)
            df.loc[idx, "days_after_holiday"] = min(days_after, 99)

    # 기존 v.6 스타일의 pre/post peak도 유지
    holiday_pd = pd.to_datetime(list(holiday_set))

    pre_holiday_map = {}
    for hd in holiday_pd:
        found_days = []
        offset = 1

        while len(found_days) < 2 and offset <= 7:
            prev_day = hd - pd.Timedelta(days=offset)

            if prev_day.weekday() not in [5, 6]:
                found_days.append(prev_day.normalize())

            offset += 1

        if len(found_days) >= 1:
            pre_holiday_map[found_days[0]] = 1
        if len(found_days) >= 2:
            if (found_days[0] - found_days[1]).days == 1:
                pre_holiday_map[found_days[1]] = 2

    post_holiday_map = {}
    for hd in holiday_pd:
        offset = 1

        while offset <= 7:
            next_day = hd + pd.Timedelta(days=offset)

            if next_day.weekday() not in [5, 6]:
                post_holiday_map[next_day.normalize()] = 1
                break

            offset += 1

    df["pre_holiday_peak"] = (
        df["날짜"]
        .map(pre_holiday_map)
        .fillna(0)
        .astype(int)
    )

    df["post_holiday_peak"] = (
        df["날짜"]
        .map(post_holiday_map)
        .fillna(0)
        .astype(int)
    )

    df.loc[df["holiday"] == 1, "pre_holiday_peak"] = 0
    df.loc[df["holiday"] == 1, "post_holiday_peak"] = 0

    return df


def add_seller_closed_features(df, seller_closed_dates=None):
    """
    셀러별 휴무일 feature 생성.
    미래 예측에서 이 값이 1이면 rule-based block 로직이 적용됨.
    """
    df = df.copy()

    seller_to_col = {
        "zigzag": "is_zigzag_closed",
        "kakao": "is_kakao_closed",
        "gonggu": "is_gonggu_closed",
        "naver": "is_naver_closed",
        "other": "is_other_closed",
    }

    for col in seller_to_col.values():
        df[col] = 0

    if not seller_closed_dates:
        return df

    for seller, dates in seller_closed_dates.items():
        if seller not in seller_to_col:
            continue

        closed_dates = pd.to_datetime(dates).normalize()
        col = seller_to_col[seller]

        df.loc[df["날짜"].isin(closed_dates), col] = 1

    return df


def preprocess_features(
    df,
    manual_holidays=None,
    seller_closed_dates=None,
    manual_spikes_dict=None,
    event_periods=None,
):
    df = df.copy()
    df["날짜"] = pd.to_datetime(df["날짜"], errors="coerce").dt.normalize()
    df = df.sort_values("날짜").reset_index(drop=True)

    # ------------------------------------------------------
    # 순수_기타물량이 없으면 residual로 생성
    # ------------------------------------------------------
    needed_for_other = [TOTAL, ZIG_SAME, ZIG_NEXT, KAKAO, GONGGU]

    if OTHER_PURE not in df.columns and all(c in df.columns for c in needed_for_other):
        df[OTHER_PURE] = (
            pd.to_numeric(df[TOTAL], errors="coerce").fillna(0)
            - pd.to_numeric(df[ZIG_SAME], errors="coerce").fillna(0)
            - pd.to_numeric(df[ZIG_NEXT], errors="coerce").fillna(0)
            - pd.to_numeric(df[KAKAO], errors="coerce").fillna(0)
            - pd.to_numeric(df[GONGGU], errors="coerce").fillna(0)
            - pd.to_numeric(df[NAVER_COMBINED], errors="coerce").fillna(0)
        ).clip(lower=0)

    # ------------------------------------------------------
    # 1. 날짜 피처
    # ------------------------------------------------------
    df["month"] = df["날짜"].dt.month
    df["dow"] = df["날짜"].dt.weekday
    df["weekend"] = df["dow"].isin([5, 6]).astype(int)
    df["week_of_year"] = df["날짜"].apply(lambda x: x.isocalendar().week)

    df["month"] = pd.Categorical(df["month"], categories=list(range(1, 13)))
    df["dow"] = pd.Categorical(df["dow"], categories=list(range(0, 7)))

    month_dummies = pd.get_dummies(df["month"], prefix="month").astype(int)
    dow_dummies = pd.get_dummies(df["dow"], prefix="dow").astype(int)

    df = pd.concat([df, month_dummies, dow_dummies], axis=1)

    df["month"] = df["month"].astype(int)
    df["dow"] = df["dow"].astype(int)

    # ------------------------------------------------------
    # 2. 이벤트 피처
    # ------------------------------------------------------
    if event_periods is None:
        event_periods = [
            ("2024-03-25", "2024-04-07"),
            ("2024-06-10", "2024-06-17"),
            ("2024-06-24", "2024-07-07"),
            ("2024-09-23", "2024-10-06"),
            ("2024-11-18", "2024-12-01"),
            ("2025-03-24", "2025-04-07"),
            ("2025-06-01", "2025-06-30"),
            ("2025-09-22", "2025-10-05"),
            ("2025-10-21", "2025-10-21"),
            ("2025-11-17", "2025-12-02"),
            ("2026-03-23", "2026-04-06"), # 직잿팟
            ("2026-05-18", "2026-05-31"), # 지그재그 11주년 행사
            ("2026-06-15", "2026-06-29"), # 직잿팟
            ("2026-09-28", "2026-10-12"), # 직잿팟
        ]

    df["is_event"] = 0

    for start, end in event_periods:
        start_dt = pd.to_datetime(start)
        end_dt = pd.to_datetime(end)
        mask = (df["날짜"] >= start_dt) & (df["날짜"] <= end_dt)
        df.loc[mask, "is_event"] = 1

    # 이벤트 block/day index
    df["event_day_index"] = 0
    df["event_block_id"] = 0

    is_event = df["is_event"].eq(1)
    block_id = is_event.ne(is_event.shift()).cumsum()

    df.loc[is_event, "event_block_id"] = block_id[is_event].astype(int)
    df.loc[is_event, "event_day_index"] = (
        df[is_event]
        .groupby(block_id[is_event])
        .cumcount()
        + 1
    )

    df["event_after_1d"] = df["is_event"].shift(1).fillna(0).astype(int)
    df["event_after_2d"] = df["is_event"].shift(2).fillna(0).astype(int)
    df["event_before_1d"] = df["is_event"].shift(-1).fillna(0).astype(int)

    # ------------------------------------------------------
    # 3. 공휴일 피처
    # ------------------------------------------------------
    years = sorted(df["날짜"].dt.year.dropna().unique().tolist())
    holiday_set = build_holiday_set(years, manual_holidays=manual_holidays)

    # Timestamp vs date 타입 문제 방지: 반드시 .dt.date로 비교
    df["holiday"] = df["날짜"].dt.date.isin(holiday_set).astype(int)

    # 주말은 weekend로 따로 잡히므로 holiday에서는 제거
    df.loc[df["weekend"] == 1, "holiday"] = 0

    df = add_pre_post_holiday_features(df, holiday_set)

    # ------------------------------------------------------
    # 4. 셀러별 휴무 피처
    # ------------------------------------------------------
    df = add_seller_closed_features(
        df,
        seller_closed_dates=seller_closed_dates,
    )

    # ------------------------------------------------------
    # 5. 자동 spike 탐지
    # ------------------------------------------------------
    auto_cols = [c for c in ALL_TARGETS if c in df.columns]

    for col in auto_cols:
        df[f"{col}_spike_up_mild"] = 0
        df[f"{col}_spike_up_strong"] = 0
        df[f"{col}_spike_down"] = 0

        trend_col = f"{col}_trend"
        resid_col = f"{col}_resid_ratio"

        df[trend_col] = np.nan
        df[resid_col] = np.nan

        for d in range(7):
            idx_d = df.index[df["dow"] == d]
            if len(idx_d) == 0:
                continue

            s = pd.to_numeric(df.loc[idx_d, col], errors="coerce").sort_index()
            baseline = s.shift(1).rolling(window=3, min_periods=2).mean()
            resid = (s - baseline) / (baseline + 1e-5)

            df.loc[idx_d, trend_col] = baseline.astype(float).values
            df.loc[idx_d, resid_col] = resid.astype(float).values

        for wk_flag in [0, 1]:
            if wk_flag == 0:
                upper1, upper2, lower1 = 0.10, 0.20, -0.15
            else:
                upper1, upper2, lower1 = 0.20, 0.25, -0.10

            mask_up_mild = (
                (df[resid_col] >= upper1)
                & (df[resid_col] < upper2)
                & (df["weekend"] == wk_flag)
            )
            mask_up_strong = (
                (df[resid_col] >= upper2)
                & (df["weekend"] == wk_flag)
            )
            mask_down = (
                (df[resid_col] <= lower1)
                & (df["weekend"] == wk_flag)
            )

            df.loc[mask_up_mild, f"{col}_spike_up_mild"] = 1
            df.loc[mask_up_strong, f"{col}_spike_up_strong"] = 1
            df.loc[mask_down, f"{col}_spike_down"] = 1

    # ------------------------------------------------------
    # 6. 수동 spike 지정
    # ------------------------------------------------------
    if manual_spikes_dict:
        for tgt, spikes in manual_spikes_dict.items():
            for spike_type, dates in spikes.items():
                if dates:
                    dts = pd.to_datetime(dates).normalize()
                    spike_col = f"{tgt}_spike_{spike_type}"

                    if spike_col not in df.columns:
                        df[spike_col] = 0

                    df.loc[df["날짜"].isin(dts), spike_col] = 1

    # ------------------------------------------------------
    # 7. 기온 피처
    # ------------------------------------------------------
    if "avg_temp" in df.columns:
        df = add_temp_shift_features(df, temp_col="avg_temp")
        df = df.drop(columns=["avg_temp"], errors="ignore")

    # ------------------------------------------------------
    # 8. 각 타겟 lag/rolling 생성
    # ------------------------------------------------------
    target_closed_map = {
        ZIG_SAME: "is_zigzag_closed",
        ZIG_NEXT: "is_zigzag_closed",
        KAKAO: "is_kakao_closed",
        GONGGU: "is_gonggu_closed",
        NAVER_COMBINED: "is_naver_closed",
        OTHER_PURE: "is_other_closed",
    }

    for tgt in ALL_TARGETS:
        # 해당 타겟의 휴무일 또는 전체 법정 공휴일을 보간(Impute) 마스크로 지정
        if tgt in target_closed_map:
            closed_col = target_closed_map[tgt]
            impute_mask = (df[closed_col] == 1) | (df["holiday"] == 1)
        else:
            impute_mask = (df["holiday"] == 1)

        df = add_target_lag_rolling_features(
            df,
            target_col=tgt,
            lags=(1, 7, 14, 21, 28),
            roll_windows=(7, 14, 21, 28),
            impute_mask=impute_mask, # 보간 마스크 전달
        )

    # trend/resid는 spike 탐지용이므로 학습에서는 제거
    drop_cols = []
    for col in auto_cols:
        drop_cols += [f"{col}_trend", f"{col}_resid_ratio"]

    df = df.drop(columns=drop_cols, errors="ignore")
    
    return df 

# ==========================================================
# 2. XGBoost Helper
# ==========================================================
def weighted_mse_alpha(alpha=1.5):
    """
    과소예측을 더 강하게 벌점 주는 custom objective.
    label - pred > 0이면 실제가 예측보다 큰 것, 즉 과소예측.
    """
    def obj(label, pred):
        residual = label - pred
        grad = np.where(residual > 0, -2 * alpha * residual, -2 * residual)
        hess = np.where(residual > 0, 2 * alpha, 2.0)
        return grad, hess

    return obj


def _ensure_cols(df, cols, fill_value=0.0):
    out = df.copy()
    missing = [c for c in cols if c not in out.columns]

    for c in missing:
        out[c] = fill_value

    return out[cols]


def _valid_target_mask(df, target_col, zero_run_min=3):
    """
    오래 이어지는 0은 실제 휴무/데이터 결측/운영 중단 가능성이 있으므로
    학습에서 제외하기 위한 mask.
    """
    s = pd.to_numeric(df[target_col], errors="coerce")
    base_mask = s.notna() & (s > 0)

    is_zero = s.fillna(0).eq(0)
    group_id = is_zero.ne(is_zero.shift()).cumsum()
    zero_run_len = is_zero.groupby(group_id).transform("sum")

    long_zero_mask = is_zero & (zero_run_len >= zero_run_min)
    return base_mask & (~long_zero_mask)


def _clean_lag_features_from_fake_zero(df, target_cols, zero_run_min=3):
    """
    긴 0 구간이 lag/rolling에 그대로 들어가면 미래 예측을 과하게 낮출 수 있으므로
    feature 생성 전 NaN으로 바꿔 rolling 계산에서 영향 완화.
    """
    out = df.copy()

    for col in target_cols:
        if col not in out.columns:
            continue

        s = pd.to_numeric(out[col], errors="coerce")
        is_zero = s.fillna(0).eq(0)
        group_id = is_zero.ne(is_zero.shift()).cumsum()
        zero_run_len = is_zero.groupby(group_id).transform("sum")

        fake_zero_mask = is_zero & (zero_run_len >= zero_run_min)
        out.loc[fake_zero_mask, col] = np.nan

    return out


def _overwrite_ts_features(row_df, tgt, buf):
    row = row_df.copy()

    # [수정] 0값을 지우거나 변환하지 않고, 있는 그대로 사용하여 주말 패턴 유지
    for lag in (1, 7, 14, 21, 28):
        row[f"{tgt}_lag_{lag}"] = buf[-lag] if len(buf) >= lag else 0.0

    for w in (7, 14, 21, 28):
        tail = buf[-w:]
        row[f"{tgt}_roll_mean_{w}"] = float(np.mean(tail)) if len(tail) >= 2 else 0.0
        row[f"{tgt}_roll_std_{w}"] = float(np.std(tail, ddof=1)) if len(tail) >= 2 else 0.0

    return row


def apply_seller_block_postprocess(row, tgt, pred_i, other_closed_ratio=0.5):
    """
    셀러별 휴무일 강제 block.
    - 지그재그 휴무: 지그재그_당일/익일 = 0
    - 카카오 휴무: 카카오모빌리티 = 0
    - 공구우먼 휴무: 공구우먼 = 0
    - 기타 휴무: 순수_기타물량은 비율 감소 옵션 적용
    """
    pred_i = float(pred_i)

    if tgt in [ZIG_SAME, ZIG_NEXT]:
        if int(row.get("is_zigzag_closed", pd.Series([0])).iloc[0]) == 1:
            return 0.0

    if tgt == KAKAO:
        if int(row.get("is_kakao_closed", pd.Series([0])).iloc[0]) == 1:
            return 0.0

    if tgt == GONGGU:
        if int(row.get("is_gonggu_closed", pd.Series([0])).iloc[0]) == 1:
            return 0.0
        
    if tgt == NAVER_COMBINED:
        if int(row.get("is_naver_closed", pd.Series([0])).iloc[0]) == 1:
            return 0.0

    if tgt == OTHER_PURE:
        if int(row.get("is_other_closed", pd.Series([0])).iloc[0]) == 1:
            return max(0.0, pred_i * other_closed_ratio)

    return max(0.0, pred_i)


def build_feature_dict(full_df):
    """
    Target leakage 방지용 feature set 구성.
    모든 raw target, 당일 집계 컬럼, 로깅용 실적 컬럼 제거.
    단, 해당 target의 lag/rolling/spike는 사용 가능.
    """
    all_cols = full_df.columns.tolist()
    features_dict = {}

    leakage_cols = [
        "날짜",
        TOTAL,
        ZIG_SAME,
        ZIG_NEXT,
        KAKAO,
        GONGGU,
        NAVER_COMBINED,
        OTHER_PURE,

        # 기존/로깅/당일 집계 컬럼들
        "지그재그 당일배송",
        "other_seller",
        "기타_당일",
        "기타_익일",
        "기타_물량",
        "기타_총물량",
        "당일_물량",
        "익일_물량",
        "셀러수",
    ]

    for tgt in BASE_TARGETS:
        exclude_other_target_features = []

        for other in ALL_TARGETS:
            if other == tgt:
                continue

            exclude_other_target_features += [
                c for c in all_cols
                if c.startswith(f"{other}_lag_")
                or c.startswith(f"{other}_roll_")
                or c.startswith(f"{other}_spike_")
            ]

        features_dict[tgt] = [
            c for c in all_cols
            if c not in leakage_cols + exclude_other_target_features
        ]

    # Meta 모델은 raw target은 금지.
    # 대신 recursive로 만들 수 있는 TOTAL lag/rolling, 일반 feature, base 예측값은 사용.
    meta_leakage_cols = [
        "날짜",
        TOTAL,
        ZIG_SAME,
        ZIG_NEXT,
        KAKAO,
        GONGGU,
        NAVER_COMBINED,
        OTHER_PURE,
        "지그재그 당일배송",
        "other_seller",
        "기타_당일",
        "기타_익일",
        "기타_물량",
        "기타_총물량",
        "당일_물량",
        "익일_물량",
        "셀러수",
    ]

    # Meta에서 base target의 lag/rolling은 제외하고, TOTAL lag/rolling만 남김
    exclude_base_ts_for_meta = []
    for base_tgt in BASE_TARGETS:
        exclude_base_ts_for_meta += [
            c for c in all_cols
            if c.startswith(f"{base_tgt}_lag_")
            or c.startswith(f"{base_tgt}_roll_")
            or c.startswith(f"{base_tgt}_spike_")
        ]

    features_dict[TOTAL] = [
        c for c in all_cols
        if c not in meta_leakage_cols + exclude_base_ts_for_meta
    ]

    return features_dict


# ==========================================================
# 3. 학습/검증/미래 예측 함수
# ==========================================================
def train_validate_forecast(
    train,
    forecast_days=7,
    valid_days=7,
    temp_future_df=None,
    manual_holidays=None,
    seller_closed_dates=None,
    manual_spikes_train=None,
    manual_spikes_future=None,
    manual_seller_fixed_volumes=None,
    base_recent_window=150,
    meta_recent_window=240,
    meta_blend_weight=0.2,
    other_closed_ratio=0.5,
    debug_future_features=True,
):
    RANDOM_STATE = 42
    MIN_TRAIN_SAMPLES = 30
    ZERO_RUN_MIN = 3

    train_sorted = train.copy()
    train_sorted["날짜"] = pd.to_datetime(train_sorted["날짜"], errors="coerce").dt.normalize()
    train_sorted = train_sorted.sort_values("날짜").reset_index(drop=True)

    # 순수 기타 타겟 재계산
    if all(c in train_sorted.columns for c in [TOTAL, ZIG_SAME, ZIG_NEXT, KAKAO, GONGGU, NAVER_COMBINED]):
        train_sorted[OTHER_PURE] = (
            pd.to_numeric(train_sorted[TOTAL], errors="coerce").fillna(0)
            - pd.to_numeric(train_sorted[ZIG_SAME], errors="coerce").fillna(0)
            - pd.to_numeric(train_sorted[ZIG_NEXT], errors="coerce").fillna(0)
            - pd.to_numeric(train_sorted[KAKAO], errors="coerce").fillna(0)
            - pd.to_numeric(train_sorted[GONGGU], errors="coerce").fillna(0)
            - pd.to_numeric(train_sorted[NAVER_COMBINED], errors="coerce").fillna(0)
        ).clip(lower=0)

    train_for_feature = _clean_lag_features_from_fake_zero(
        train_sorted,
        target_cols=BASE_TARGETS,
        zero_run_min=ZERO_RUN_MIN,
    )

    train_feat_all = preprocess_features(
        train_for_feature,
        manual_holidays=manual_holidays,
        seller_closed_dates=seller_closed_dates,
        manual_spikes_dict=manual_spikes_train,
    )

    split_idx = len(train_feat_all) - valid_days
    train_df = train_feat_all.iloc[:split_idx].copy()
    valid_df = train_feat_all.iloc[split_idx:].copy()
    full_df = train_feat_all.copy()

    last_date = train_sorted["날짜"].max()
    future_dates = pd.date_range(
        start=last_date + pd.Timedelta(days=1),
        periods=forecast_days,
    )

    future_df = pd.DataFrame({"날짜": future_dates})

    if temp_future_df is not None:
        temp_future_df = temp_future_df.copy()
        temp_future_df["날짜"] = pd.to_datetime(temp_future_df["날짜"], errors="coerce").dt.normalize()

        future_temp_input = pd.merge(
            future_df,
            temp_future_df[["날짜", "avg_temp"]],
            on="날짜",
            how="left",
        )

        # 예보가 비는 날짜는 앞뒤 값 또는 15도로 대체
        future_temp_input["avg_temp"] = (
            future_temp_input["avg_temp"]
            .ffill()
            .bfill()
            .fillna(15.0)
        )
    else:
        future_temp_input = future_df.copy()
        future_temp_input["avg_temp"] = 15.0

    future_feat_df = preprocess_features(
        future_temp_input,
        manual_holidays=manual_holidays,
        seller_closed_dates=seller_closed_dates,
        manual_spikes_dict=manual_spikes_future,
    )

    future_feat_df = future_feat_df.iloc[-forecast_days:].reset_index(drop=True)

    for tgt in ALL_TARGETS:
        if tgt not in future_feat_df.columns:
            future_feat_df[tgt] = np.nan

    if debug_future_features:
        print("\n=== 미래 feature 체크 ===")
        debug_cols = [
            "날짜",
            "dow",
            "weekend",
            "holiday",
            "pre_holiday_peak",
            "post_holiday_peak",
            "days_until_holiday",
            "days_after_holiday",
            "is_zigzag_closed",
            "is_kakao_closed",
            "is_gonggu_closed",
            "is_naver_closed",
            "is_other_closed",
        ]
        display(future_feat_df[[c for c in debug_cols if c in future_feat_df.columns]])

    features_dict = build_feature_dict(full_df)

    xgb_base_params = {
        "n_estimators": 900,
        "learning_rate": 0.015,
        "max_depth": 3,
        "min_child_weight": 5,
        "gamma": 0.05,
        "subsample": 1.0,
        "colsample_bytree": 1.0,
        "reg_alpha": 0.1,
        "reg_lambda": 3.0,
        "objective": weighted_mse_alpha(alpha=1.6),
        "random_state": RANDOM_STATE,
        "tree_method": "hist",
        "eval_metric": "rmse",
    }

    xgb_meta_params = {
        "n_estimators": 500,
        "learning_rate": 0.02,
        "max_depth": 2,
        "min_child_weight": 8,
        "gamma": 0.1,
        "subsample": 1.0,
        "colsample_bytree": 1.0,
        "reg_alpha": 0.2,
        "reg_lambda": 5.0,
        "objective": weighted_mse_alpha(alpha=1.3),
        "random_state": RANDOM_STATE,
        "tree_method": "hist",
        "eval_metric": "rmse",
    }

    trained_models = {}
    pred_train_base = {}
    pred_valid_base = {}
    pred_full_base = {}

    print("\n=== Base 모델 학습 제외 데이터 확인 ===")

    for tgt in BASE_TARGETS:
        train_recent = train_df.tail(base_recent_window).copy()
        full_recent = full_df.tail(base_recent_window).copy()

        X_tr = _ensure_cols(train_recent, features_dict[tgt]).fillna(0.0)
        y_tr = pd.to_numeric(train_recent[tgt], errors="coerce")

        X_val = _ensure_cols(valid_df, features_dict[tgt]).fillna(0.0)

        X_train_all = _ensure_cols(train_df, features_dict[tgt]).fillna(0.0)
        X_full_all = _ensure_cols(full_df, features_dict[tgt]).fillna(0.0)

        X_full_recent = _ensure_cols(full_recent, features_dict[tgt]).fillna(0.0)
        y_full_recent = pd.to_numeric(full_recent[tgt], errors="coerce")

        mask_tr = _valid_target_mask(train_recent, tgt, ZERO_RUN_MIN)
        mask_full = _valid_target_mask(full_recent, tgt, ZERO_RUN_MIN)

        print(
            f"{tgt}: train 최근 {len(train_recent)}건 중 사용 {mask_tr.sum()}건 / 제외 {len(mask_tr) - mask_tr.sum()}건, "
            f"full 최근 {len(full_recent)}건 중 사용 {mask_full.sum()}건 / 제외 {len(mask_full) - mask_full.sum()}건"
        )

        if mask_tr.sum() >= MIN_TRAIN_SAMPLES:
            model = XGBRegressor(**xgb_base_params)
            model.fit(
                X_tr.loc[mask_tr],
                np.log1p(y_tr.loc[mask_tr].astype(float)),
                verbose=False,
            )

            pred_train_base[tgt] = np.clip(np.expm1(model.predict(X_train_all)), 0, None)
            pred_valid_base[tgt] = np.clip(np.expm1(model.predict(X_val)), 0, None)

            model_full = XGBRegressor(**xgb_base_params)
            model_full.fit(
                X_full_recent.loc[mask_full],
                np.log1p(y_full_recent.loc[mask_full].astype(float)),
                verbose=False,
            )

            pred_full_base[tgt] = np.clip(np.expm1(model_full.predict(X_full_all)), 0, None)
            trained_models[tgt] = model_full

        else:
            recent_valid = y_full_recent.loc[mask_full].tail(28)
            mean_val = float(recent_valid.mean()) if len(recent_valid) > 0 else 0.0

            pred_train_base[tgt] = np.full(len(X_train_all), mean_val)
            pred_valid_base[tgt] = np.full(len(X_val), mean_val)
            pred_full_base[tgt] = np.full(len(X_full_all), mean_val)
            trained_models[tgt] = None

    print("====================================\n")

    # ------------------------------------------------------
    # Meta 모델 학습
    # ------------------------------------------------------
    meta_features = features_dict[TOTAL] + [f"pred_{t}" for t in BASE_TARGETS] + ["base_sum_pred"]

    X_tr_meta = _ensure_cols(train_df, features_dict[TOTAL]).fillna(0.0)
    X_val_meta = _ensure_cols(valid_df, features_dict[TOTAL]).fillna(0.0)
    X_full_meta = _ensure_cols(full_df, features_dict[TOTAL]).fillna(0.0)

    for tgt in BASE_TARGETS:
        X_tr_meta[f"pred_{tgt}"] = pred_train_base[tgt]
        X_val_meta[f"pred_{tgt}"] = pred_valid_base[tgt]
        X_full_meta[f"pred_{tgt}"] = pred_full_base[tgt]

    X_tr_meta["base_sum_pred"] = sum(X_tr_meta[f"pred_{t}"] for t in BASE_TARGETS)
    X_val_meta["base_sum_pred"] = sum(X_val_meta[f"pred_{t}"] for t in BASE_TARGETS)
    X_full_meta["base_sum_pred"] = sum(X_full_meta[f"pred_{t}"] for t in BASE_TARGETS)

    X_tr_meta = X_tr_meta[meta_features]
    X_val_meta = X_val_meta[meta_features]
    X_full_meta = X_full_meta[meta_features]

    y_tr_meta = pd.to_numeric(train_df[TOTAL], errors="coerce")
    y_full_meta = pd.to_numeric(full_df[TOTAL], errors="coerce")

    train_meta_recent_idx = train_df.tail(meta_recent_window).index
    full_meta_recent_idx = full_df.tail(meta_recent_window).index

    X_tr_meta_recent = X_tr_meta.loc[train_meta_recent_idx]
    y_tr_meta_recent = y_tr_meta.loc[train_meta_recent_idx]
    train_df_meta_recent = train_df.loc[train_meta_recent_idx]

    X_full_meta_recent = X_full_meta.loc[full_meta_recent_idx]
    y_full_meta_recent = y_full_meta.loc[full_meta_recent_idx]
    full_df_meta_recent = full_df.loc[full_meta_recent_idx]

    mask_tr_meta = _valid_target_mask(train_df_meta_recent, TOTAL, ZERO_RUN_MIN)
    mask_full_meta = _valid_target_mask(full_df_meta_recent, TOTAL, ZERO_RUN_MIN)

    print(
        f"{TOTAL}: meta 최근 {len(train_df_meta_recent)}건 중 사용 {mask_tr_meta.sum()}건 / 제외 {len(mask_tr_meta) - mask_tr_meta.sum()}건"
    )

    model_meta = XGBRegressor(**xgb_meta_params)
    model_meta.fit(
        X_tr_meta_recent.loc[mask_tr_meta],
        np.log1p(y_tr_meta_recent.loc[mask_tr_meta].astype(float)),
        verbose=False,
    )

    meta_valid_pred = np.clip(np.expm1(model_meta.predict(X_val_meta)), 0, None)
    base_sum_valid_pred = X_val_meta["base_sum_pred"].values

    pred_valid_total = (
        (1 - meta_blend_weight) * base_sum_valid_pred
        + meta_blend_weight * meta_valid_pred
    )

    model_meta_full = XGBRegressor(**xgb_meta_params)
    model_meta_full.fit(
        X_full_meta_recent.loc[mask_full_meta],
        np.log1p(y_full_meta_recent.loc[mask_full_meta].astype(float)),
        verbose=False,
    )

    trained_models[TOTAL] = model_meta_full

    # ------------------------------------------------------
    # 검증 성능 출력
    # ------------------------------------------------------
    print("\n=== 검증 성능 (MAPE) ===")

    actual_total = pd.to_numeric(valid_df[TOTAL], errors="coerce").values
    mask_total = actual_total > 0

    if mask_total.sum() > 0:
        mape_total = mean_absolute_percentage_error(
            actual_total[mask_total],
            pred_valid_total[mask_total],
        )
        print(f"총 물량: {mape_total:.4f} (정확도 {100 * (1 - mape_total):.2f}%)")

    for tgt in BASE_TARGETS:
        actuals = pd.to_numeric(valid_df[tgt], errors="coerce").values
        preds = pred_valid_base[tgt]
        mask = actuals > 0

        if mask.sum() > 0:
            mape = mean_absolute_percentage_error(actuals[mask], preds[mask])
            print(f"{tgt}: {mape:.4f} (정확도 {100 * (1 - mape):.2f}%)")

    print("========================\n")

    # ------------------------------------------------------
    # 미래 Recursive Forecasting
    # ------------------------------------------------------
    target_closed_map = {
        ZIG_SAME: "is_zigzag_closed",
        ZIG_NEXT: "is_zigzag_closed",
        KAKAO: "is_kakao_closed",
        GONGGU: "is_gonggu_closed",
        NAVER_COMBINED: "is_naver_closed",
        OTHER_PURE: "is_other_closed",
    }

    bufs = {}
    for tgt in ALL_TARGETS:
        s_buf = pd.to_numeric(full_df[tgt], errors="coerce").copy()
        
        # [수정] 모든 0이 아닌, '실제 휴무일/공휴일'만 콕 집어서 7일 전 데이터로 채움
        if tgt in target_closed_map:
            closed_col = target_closed_map[tgt]
            impute_mask = (full_df[closed_col] == 1) | (full_df["holiday"] == 1)
        else:
            impute_mask = (full_df["holiday"] == 1)
            
        s_buf.loc[impute_mask] = np.nan
        s_buf = s_buf.fillna(s_buf.shift(7))
        s_buf = s_buf.interpolate(method="linear").bfill().ffill().fillna(0)
        
        bufs[tgt] = s_buf.tolist()

    future_preds = {tgt: [] for tgt in ALL_TARGETS}

    for i in range(forecast_days):
        row = future_feat_df.iloc[[i]].copy()
        cur_date_str = row["날짜"].dt.strftime("%Y-%m-%d").iloc[0] # 현재 날짜 문자열 추출

        for tgt in BASE_TARGETS:
            row = _overwrite_ts_features(row, tgt, bufs[tgt])
            X_row = _ensure_cols(row, features_dict[tgt]).fillna(0.0)

            if trained_models[tgt] is not None:
                pred_i = float(np.expm1(trained_models[tgt].predict(X_row))[0])
                pred_i = max(0.0, pred_i)
            else:
                recent_buf = [
                    x
                    for x in bufs[tgt]
                    if pd.notna(x) and float(x) > 0
                ]
                pred_i = float(np.mean(recent_buf[-7:])) if len(recent_buf) > 0 else 0.0

            # 핵심: 셀러별 휴무일 block 후처리
            pred_i = apply_seller_block_postprocess(
                row=row,
                tgt=tgt,
                pred_i=pred_i,
                other_closed_ratio=other_closed_ratio,
            )

            # 확정된 물량은 덮어쓰기
            if manual_seller_fixed_volumes and tgt in manual_seller_fixed_volumes:
                if cur_date_str in manual_seller_fixed_volumes[tgt]:
                    pred_i = float(manual_seller_fixed_volumes[tgt][cur_date_str])

            future_preds[tgt].append(pred_i)
            bufs[tgt].append(pred_i)
            row[f"pred_{tgt}"] = pred_i

        base_sum_pred = sum(row[f"pred_{t}"].iloc[0] for t in BASE_TARGETS)
        row["base_sum_pred"] = base_sum_pred

        row = _overwrite_ts_features(row, TOTAL, bufs[TOTAL])
        X_row_meta = _ensure_cols(row, meta_features).fillna(0.0)

        meta_pred = max(
            0.0,
            float(np.expm1(trained_models[TOTAL].predict(X_row_meta))[0]),
        )

        pred_total = (
            (1 - meta_blend_weight) * base_sum_pred
            + meta_blend_weight * meta_pred
        )

        future_preds[TOTAL].append(pred_total)
        bufs[TOTAL].append(pred_total)

    # ------------------------------------------------------
    # 결과 테이블 구성
    # ------------------------------------------------------
    valid_result_df = pd.DataFrame({
        "날짜": valid_df["날짜"].dt.strftime("%Y-%m-%d"),
    })

    valid_result_df["실제_총물량"] = pd.to_numeric(valid_df[TOTAL], errors="coerce").values
    valid_result_df["예측_총물량"] = np.round(pred_valid_total).astype(int)
    valid_result_df["base_sum_예측"] = np.round(base_sum_valid_pred).astype(int)
    valid_result_df["meta_예측"] = np.round(meta_valid_pred).astype(int)

    for tgt in BASE_TARGETS:
        valid_result_df[f"실제_{tgt}"] = pd.to_numeric(valid_df[tgt], errors="coerce").values
        valid_result_df[f"예측_{tgt}"] = np.round(pred_valid_base[tgt]).astype(int)

    forecast_df = pd.DataFrame({
        "날짜": future_feat_df["날짜"].dt.strftime("%Y-%m-%d"),
    })

    for tgt in BASE_TARGETS:
        forecast_df[tgt] = np.round(future_preds[tgt]).astype(int)

    forecast_df["base_sum"] = forecast_df[BASE_TARGETS].sum(axis=1)
    forecast_df["총 물량"] = np.round(future_preds[TOTAL]).astype(int)

    # 휴무일 feature도 결과에 같이 표시해서 block 확인 가능하게 함
    closed_cols = [
        "is_zigzag_closed",
        "is_kakao_closed",
        "is_gonggu_closed",
        "is_naver_closed",
        "is_other_closed",
        "holiday",
        "pre_holiday_peak",
        "post_holiday_peak",
    ]

    for col in closed_cols:
        if col in future_feat_df.columns:
            forecast_df[col] = future_feat_df[col].values

    return trained_models, forecast_df, valid_result_df, features_dict

### Prediction

In [40]:
# ==========================================================
# [v.6.4.1] Cell 3. 공통 환경 설정 (공휴일 / 휴무일 / 화주사 확정 물량)
# ==========================================================

# 1. 날짜 및 기온 데이터 형식 정리
data["날짜"] = pd.to_datetime(data["날짜"], errors="coerce").dt.normalize()
temp_merge_data["날짜"] = pd.to_datetime(temp_merge_data["날짜"], errors="coerce").dt.normalize()
data["avg_temp"] = pd.to_numeric(data["avg_temp"], errors="coerce")
temp_merge_data["avg_temp"] = pd.to_numeric(temp_merge_data["avg_temp"], errors="coerce")

# ----------------------------------------------------------
# 2. 수동 공휴일 설정
# ----------------------------------------------------------
manual_holidays = [
    "2026-05-01",  # 근로자의 날
]

# ----------------------------------------------------------
# 3. 셀러별 휴무일 설정
# ----------------------------------------------------------
seller_closed_dates = {
    "zigzag": [],
    "kakao": [],
    "gonggu": ["2026-05-01"],
    "naver": [],  # 💡 6.4.1은 네이버가 하나로 통합됨
    "other": ["2026-05-01"],
}

# ----------------------------------------------------------
# 4. 화주사 제공 확정 물량 (하드코딩)
#    - 평상시: {} 로 비워두기
#    - 행사시: {"YYYY-MM-DD": 물량} 입력
#    - 키는 타겟 상수로 지정 (ZIG_SAME, ZIG_NEXT, NAVER_COMBINED, KAKAO ...)
# ----------------------------------------------------------
manual_seller_fixed_volumes = {
    ZIG_SAME:       {},  # 예) {"2026-10-21": 11059, "2026-10-22": 11202}
    ZIG_NEXT:       {},
    NAVER_COMBINED: {"2026-10-28": 16900, "2026-10-29": 16600, "2026-10-30": 15900, "2026-10-31": 13600, "2026-11-01": 9600, "2026-11-02": 14300, "2026-11-03": 15600},  
}
fixed_volumes = {k: v for k, v in manual_seller_fixed_volumes.items() if v}  # 값 있는 타겟만

# ----------------------------------------------------------
# 5. 학습 데이터 강제 스파이크 (보통 None 유지)
# ----------------------------------------------------------
manual_spikes_train = None

# ----------------------------------------------------------
# 6. 모든 시나리오 공통 실행 파라미터
# ----------------------------------------------------------
COMMON_KWARGS = dict(
    train=data,
    forecast_days=7,
    valid_days=7,
    temp_future_df=temp_merge_data,
    manual_holidays=manual_holidays,
    seller_closed_dates=seller_closed_dates,
    manual_spikes_train=manual_spikes_train,
    base_recent_window=150,
    meta_recent_window=240,
    meta_blend_weight=0.2,
    other_closed_ratio=0.5,
    debug_future_features=False,
)

print("✅ [v.6.4.1] Cell 3 환경 설정 완료!")
print(f"   확정 물량 입력 타겟: {list(fixed_volumes.keys()) or '없음'}")

✅ [v.6.4.1] Cell 3 환경 설정 완료!
   확정 물량 입력 타겟: ['네이버_통합']


In [41]:
# ==========================================================
# [v.6.4.1] Cell 4. 시나리오 자동 조합 실행 + 비교 리포트 (저장 안 함)
# ==========================================================
from itertools import product

# ----------------------------------------------------------
# 1. 조합 설정 (보통 손댈 필요 없음)
# ----------------------------------------------------------
# 스파이크 적용 날짜: None이면 예측 7일 전체에 자동 적용
SPIKE_DATES = None   # 특정 날짜만 원하면 ["2026-10-30", "2026-10-31"]

# 스파이크 강도
SPIKE_LEVELS = ["up_mild", "up_strong", "down"]

# 스파이크를 줄 타겟 묶음 (불필요한 묶음은 주석 처리하면 실행 수가 줄어듦)
SPIKE_GROUPS = {
    "지그재그": [ZIG_SAME, ZIG_NEXT],
    "네이버":   [NAVER_COMBINED],
    "전체":     BASE_TARGETS + [TOTAL],
}

# 확정 물량 반영 여부: Cell 3에 값이 있으면 모든 조합에 대해 On/Off 둘 다 실행
FIXED_OPTIONS = [False, True] if fixed_volumes else [False]

# ----------------------------------------------------------
# 2. 예측 7일 날짜 자동 계산 (Cell 2와 동일한 방식: 학습 마지막 날 + 1일부터)
# ----------------------------------------------------------
_last = pd.to_datetime(data["날짜"]).max()
forecast_dates = pd.date_range(_last + pd.Timedelta(days=1), periods=COMMON_KWARGS["forecast_days"])
forecast_date_strs = forecast_dates.strftime("%Y-%m-%d").tolist()
spike_dates = SPIKE_DATES or forecast_date_strs
print(f" 예측 기간: {forecast_date_strs[0]} ~ {forecast_date_strs[-1]}")
print(f"⚡ 스파이크 적용일: {spike_dates}")

# ----------------------------------------------------------
# 3. 시나리오 자동 생성
# ----------------------------------------------------------
spike_variants = {"Base": {}}
for (g_name, g_targets), level in product(SPIKE_GROUPS.items(), SPIKE_LEVELS):
    spike_variants[f"{g_name}_{level}"] = {t: {level: spike_dates} for t in g_targets}

scenarios = {}
for (s_name, spikes), use_fixed in product(spike_variants.items(), FIXED_OPTIONS):
    name = f"{s_name}+확정" if use_fixed else s_name
    scenarios[name] = {"spikes": spikes, "fixed": use_fixed}

print(f" 총 {len(scenarios)}개 시나리오 실행 예정\n")

# 확정 날짜가 예측 기간 밖이면 경고
for tgt, d in fixed_volumes.items():
    out = [x for x in d if x not in forecast_date_strs]
    if out:
        print(f"⚠️ [{tgt}] 예측 기간 밖 확정 날짜는 무시됨: {out}")

# ----------------------------------------------------------
# 4. 실행
# ----------------------------------------------------------
scenario_runs = {}
for name, cfg in tqdm(scenarios.items(), total=len(scenarios)):
    models, fc_df, valid_df_, feats = train_validate_forecast(
        **COMMON_KWARGS,
        manual_spikes_future=cfg["spikes"],
        manual_seller_fixed_volumes=fixed_volumes if cfg["fixed"] else None,
    )
    scenario_runs[name] = {"forecast": fc_df, "valid": valid_df_, "models": models, "features": feats}

print("\n✅ 모든 시나리오 연산 완료!\n")

# ----------------------------------------------------------
# 5. 리포트
# ----------------------------------------------------------
names = list(scenario_runs.keys())
base_fc = scenario_runs["Base"]["forecast"]
base_week = base_fc[TOTAL].sum()

# (1) 요약표: 시나리오별 7일 합계 & Base 대비 증감
summary_rows = []
for n in names:
    fc = scenario_runs[n]["forecast"]
    row = {"시나리오": n, "7일_총물량": int(fc[TOTAL].sum())}
    row["Base대비"] = row["7일_총물량"] - int(base_week)
    row["Base대비(%)"] = round(row["Base대비"] / base_week * 100, 1)
    for tgt in BASE_TARGETS:
        row[tgt] = int(fc[tgt].sum())
    summary_rows.append(row)
summary_df = pd.DataFrame(summary_rows)

print("📊 [시나리오 요약 - 7일 합계]")
display(summary_df)

# (2) 일자별 총 물량 비교
daily_total_df = base_fc[["날짜"]].copy()
for n in names:
    daily_total_df[n] = scenario_runs[n]["forecast"][TOTAL].values

print("\n📊 [일자별 총 물량 비교]")
display(daily_total_df)

print(f"\n Cell 5의 SELECTED_SCENARIO 에 위 '시나리오' 이름 중 하나를 입력하세요.")

 예측 기간: 2026-10-28 ~ 2026-11-03
⚡ 스파이크 적용일: ['2026-10-28', '2026-10-29', '2026-10-30', '2026-10-31', '2026-11-01', '2026-11-02', '2026-11-03']
 총 20개 시나리오 실행 예정



  0%|          | 0/20 [00:00<?, ?it/s]


=== Base 모델 학습 제외 데이터 확인 ===
지그재그_당일: train 최근 150건 중 사용 150건 / 제외 0건, full 최근 150건 중 사용 150건 / 제외 0건
지그재그_익일: train 최근 150건 중 사용 150건 / 제외 0건, full 최근 150건 중 사용 150건 / 제외 0건
카카오모빌리티: train 최근 150건 중 사용 149건 / 제외 1건, full 최근 150건 중 사용 150건 / 제외 0건
공구우먼: train 최근 150건 중 사용 124건 / 제외 26건, full 최근 150건 중 사용 125건 / 제외 25건
네이버_통합: train 최근 150건 중 사용 43건 / 제외 107건, full 최근 150건 중 사용 50건 / 제외 100건
순수_기타물량: train 최근 150건 중 사용 149건 / 제외 1건, full 최근 150건 중 사용 149건 / 제외 1건

총 물량: meta 최근 240건 중 사용 240건 / 제외 0건

=== 검증 성능 (MAPE) ===
총 물량: 0.1359 (정확도 86.41%)
지그재그_당일: 0.4510 (정확도 54.90%)
지그재그_익일: 0.1488 (정확도 85.12%)
카카오모빌리티: 0.1217 (정확도 87.83%)
공구우먼: 0.5324 (정확도 46.76%)
네이버_통합: 0.1025 (정확도 89.75%)
순수_기타물량: 0.1634 (정확도 83.66%)



  5%|▌         | 1/20 [00:10<03:13, 10.18s/it]


=== Base 모델 학습 제외 데이터 확인 ===
지그재그_당일: train 최근 150건 중 사용 150건 / 제외 0건, full 최근 150건 중 사용 150건 / 제외 0건
지그재그_익일: train 최근 150건 중 사용 150건 / 제외 0건, full 최근 150건 중 사용 150건 / 제외 0건
카카오모빌리티: train 최근 150건 중 사용 149건 / 제외 1건, full 최근 150건 중 사용 150건 / 제외 0건
공구우먼: train 최근 150건 중 사용 124건 / 제외 26건, full 최근 150건 중 사용 125건 / 제외 25건
네이버_통합: train 최근 150건 중 사용 43건 / 제외 107건, full 최근 150건 중 사용 50건 / 제외 100건
순수_기타물량: train 최근 150건 중 사용 149건 / 제외 1건, full 최근 150건 중 사용 149건 / 제외 1건

총 물량: meta 최근 240건 중 사용 240건 / 제외 0건

=== 검증 성능 (MAPE) ===
총 물량: 0.1359 (정확도 86.41%)
지그재그_당일: 0.4510 (정확도 54.90%)
지그재그_익일: 0.1488 (정확도 85.12%)
카카오모빌리티: 0.1217 (정확도 87.83%)
공구우먼: 0.5324 (정확도 46.76%)
네이버_통합: 0.1025 (정확도 89.75%)
순수_기타물량: 0.1634 (정확도 83.66%)



...


✅ 모든 시나리오 연산 완료!

📊 [시나리오 요약 - 7일 합계]


,시나리오,7일_총물량,Base대비,Base대비(%),지그재그_당일,지그재그_익일,카카오모빌리티,공구우먼,네이버_통합,순수_기타물량
0,Base,193920,0,0.0,42970,39077,4846,1695,83655,21986
1,Base+확정,210630,16710,8.6,42970,39077,4846,1695,102500,21986
2,지그재그_up_mild,194772,852,0.4,43967,39141,4846,1695,83655,21986
3,지그재그_up_mild+확정,211481,17561,9.1,43967,39141,4846,1695,102500,21986
4,지그재그_up_strong,200561,6641,3.4,42084,47100,4846,1695,83655,21986
5,지그재그_up_strong+확정,216586,22666,11.7,42084,47100,4846,1695,102500,21986
6,지그재그_down,177855,-16065,-8.3,28440,37844,4846,1695,83655,21986
7,지그재그_down+확정,195136,1216,0.6,28440,37844,4846,1695,102500,21986
8,네이버_up_mild,193920,0,0.0,42970,39077,4846,1695,83655,21986
9,네이버_up_mild+확정,210630,16710,8.6,42970,39077,4846,1695,102500,21986



📊 [일자별 총 물량 비교]


,날짜,Base,Base+확정,지그재그_up_mild,지그재그_up_mild+확정,지그재그_up_strong,지그재그_up_strong+확정,지그재그_down,지그재그_down+확정,네이버_up_mild,...,네이버_up_strong,네이버_up_strong+확정,네이버_down,네이버_down+확정,전체_up_mild,전체_up_mild+확정,전체_up_strong,전체_up_strong+확정,전체_down,전체_down+확정
0,2026-10-28,29668,33378,29703,33413,30393,34002,28120,31707,29668,...,30881,33378,26779,33378,29763,33475,32729,35229,23321,30135
1,2026-10-29,29666,33624,29771,33730,29963,33820,26744,31050,29666,...,31368,33624,26892,33624,29831,33792,32906,35166,22670,29636
2,2026-10-30,30244,33633,30484,33873,30052,33441,27000,30734,30244,...,31829,33633,27801,33633,30544,33930,32750,34491,22934,29279
3,2026-10-31,22339,26785,22428,26873,22037,26482,19445,23182,22339,...,24265,26785,20873,26785,22472,26922,26354,28799,17058,22258
4,2026-11-01,26212,23584,26295,23667,26211,23583,23941,21504,26212,...,27565,23584,23240,23584,26346,23713,28261,24008,20574,20725
5,2026-11-02,28759,29247,28961,29448,31183,31642,26585,27696,28759,...,29573,29247,25430,29247,29020,29506,33475,33202,22526,26281
6,2026-11-03,27032,30379,27130,30477,30722,33616,26020,29263,27032,...,28056,30379,24043,30379,27184,30537,32533,34995,21613,27890



 Cell 5의 SELECTED_SCENARIO 에 위 '시나리오' 이름 중 하나를 입력하세요.


In [42]:
# ==========================================================
# [v.6.4.1] Cell 5. 최종 시나리오 선택 → 엑셀 저장
# (Cell 4 결과를 그대로 사용하므로 재연산 없음)
# ==========================================================
import os

SELECTED_SCENARIO = "Base+확정"  # 👈 Cell 4 비교표를 보고 채택할 시나리오 이름 입력

assert SELECTED_SCENARIO in scenario_runs, f"없는 시나리오입니다: {list(scenario_runs.keys())}"
sel = scenario_runs[SELECTED_SCENARIO]

# 이후 셀(지역별 예측)에서 쓰는 변수명 유지
forecast_df_final = sel["forecast"].copy()
valid_result_df   = sel["valid"]
trained_models    = sel["models"]
features_dict     = sel["features"]

# 저장
VERSION_DIR = "./v.6.4.1"
SAVE_DIR = f"{VERSION_DIR}/Total_results"
os.makedirs(SAVE_DIR, exist_ok=True)
save_path_total = f"{SAVE_DIR}/Total_Forecasting_26_1028_1103.xlsx"  # 👈 날짜명 변경

forecast_df_final.to_excel(save_path_total, index=False)
print(f"🏁 채택 시나리오: {SELECTED_SCENARIO}")
print(f"💾 저장 완료: {save_path_total}\n")

display(forecast_df_final[["날짜", "총 물량", "base_sum"] + BASE_TARGETS])

🏁 채택 시나리오: Base+확정
💾 저장 완료: ./v.6.4.1/Total_results/Total_Forecasting_26_1028_1103.xlsx



,날짜,총 물량,base_sum,지그재그_당일,지그재그_익일,카카오모빌리티,공구우먼,네이버_통합,순수_기타물량
0,2026-10-28,33378,34056,5816,6744,885,453,16900,3258
1,2026-10-29,33624,34364,6392,6797,932,360,16600,3283
2,2026-10-30,33633,34240,6778,6861,863,412,15900,3426
3,2026-10-31,26785,26631,5782,5045,310,280,13600,1614
4,2026-11-01,23584,23784,5993,4640,175,0,9600,3376
5,2026-11-02,29247,29508,6121,4418,846,73,14300,3750
6,2026-11-03,30379,30491,6088,4572,835,117,15600,3279


# 지역별 물량 예측

In [43]:
# 최신 데이터 로딩 파이프라인
import pandas as pd
import pymysql
from sshtunnel import SSHTunnelForwarder
import os
from dotenv import load_dotenv

class AutoContainerGeneration:
    def __init__(self, version=1, debug=False):
        self.version = version
        self.debug = debug
        print(f"AutoContainerGeneration version: {version}")

        load_dotenv()

    # 공통 MySQL 데이터 추출 메서드
    def fetch_data(self, query, ssh_host, ssh_user, ssh_private_key,
                   mysql_host, mysql_port, mysql_user, mysql_password, mysql_database):
        """
        MySQL 데이터를 추출하여 DF로 변환
        """
        try:
            with SSHTunnelForwarder(
                (ssh_host, 22),
                ssh_username=ssh_user,
                ssh_private_key=ssh_private_key,
                remote_bind_address=(mysql_host, mysql_port)
            ) as tunnel:
                print("SSH 터널 연결 성공")

                with pymysql.connect(
                    host='127.0.0.1',
                    user=mysql_user,
                    passwd=mysql_password,
                    db=mysql_database,
                    charset='utf8',
                    port=tunnel.local_bind_port,
                    cursorclass=pymysql.cursors.DictCursor) as conn:
                    with conn.cursor() as cur:
                        cur.execute(query)
                        results = cur.fetchall()
                        print("쿼리 실행 완료")

                        df = pd.DataFrame(results)
                        return df

        except Exception as e:
            print(f"Error fetching data for: {e}")
            return None

    # 물량 집계 데이터 추출
    def fetch_cluster_data(self):

        # SSH 및 MySQL 설정
        ssh_host = os.getenv("SSH_HOST_VER_1")
        ssh_user = os.getenv("SSH_USER")
        ssh_private_key = os.getenv(r"SSH_PRIVATE_KEY")

        mysql_host = os.getenv("MYSQL_HOST")
        mysql_port = 3306
        mysql_user = os.getenv("MYSQL_USER")
        mysql_password = os.getenv("MYSQL_PASSWORD")
        mysql_database = os.getenv("MYSQL_DATABASE")

        print(f'Environment: {mysql_host}')

        # 실행할 쿼리
        cluster_query = """
                SELECT
                    hub_cluster_plan.plan_date AS "날짜",
                    shipping_shippingitem.id AS "주문번호",
                    shop_shop.name AS "셀러명",
                    -- 지역명 변환 로직
                    CASE 
                        WHEN ls.name LIKE '서울특별시 강남%' THEN '서울특별시 강남구'
                        WHEN RIGHT(SUBSTRING_INDEX(SUBSTRING_INDEX(ls.name, ' ', 2), ' ', -1), 1)
                            IN ('가','나','다','라','마','바','사','아','자','차','카','타','파','하')
                        THEN CONCAT(
                                SUBSTRING_INDEX(ls.name, ' ', 1), ' ',
                                IF(SUBSTRING_INDEX(ls.name, ' ', 1) = '서울특별시',
                                    CONCAT(
                                        LEFT(
                                            SUBSTRING_INDEX(SUBSTRING_INDEX(ls.name, ' ', 2), ' ', -1),
                                            CHAR_LENGTH(SUBSTRING_INDEX(SUBSTRING_INDEX(ls.name, ' ', 2), ' ', -1)) - 1
                                        ), '구'
                                    ),
                                    CONCAT(
                                        LEFT(
                                            SUBSTRING_INDEX(SUBSTRING_INDEX(ls.name, ' ', 2), ' ', -1),
                                            CHAR_LENGTH(SUBSTRING_INDEX(SUBSTRING_INDEX(ls.name, ' ', 2), ' ', -1)) - 1
                                        ), '시'
                                    )
                                )
                            )
                        ELSE ls.name
                    END AS "지역"
                FROM shipping_shippingitem
                JOIN order_order
                    ON shipping_shippingitem.order_id = order_order.id
                JOIN hub_cluster_plan_item
                    ON shipping_shippingitem.id = hub_cluster_plan_item.shipping_item_id
                JOIN hub_cluster_plan
                    ON hub_cluster_plan_item.cluster_plan_id = hub_cluster_plan.id
                JOIN location_address la
                    ON shipping_shippingitem.address_id = la.id
                JOIN location_sector ls
                    ON shipping_shippingitem.designated_sector_id = ls.id
                JOIN shop_shop
                    ON shop_shop.id = order_order.shop_id
                WHERE hub_cluster_plan.plan_date <= CURRENT_DATE()
                AND (
                    (hub_cluster_plan.plan_date < '2026-07-02' AND hub_cluster_plan.plan_no IN (1, 90)) 
                    OR 
                    (hub_cluster_plan.plan_date >= '2026-07-02' AND hub_cluster_plan.plan_no = 10)
                )
                AND ls.allowed = 1
                AND shop_shop.name != '부스터스'; """

        df_forecast = self.fetch_data(
            cluster_query,
            ssh_host, ssh_user, ssh_private_key,
            mysql_host, mysql_port, mysql_user, mysql_password, mysql_database
        )

        return df_forecast

# DaaS DB 데이터 추출
if __name__ == "__main__":
    generator = AutoContainerGeneration(version=1, debug=True)
    df_forecast = generator.fetch_cluster_data()

    # 각 DataFrame 확인
    print("최신 DF 생성 완료")

# 기존 데이터 업로드
base = pd.read_excel("./test_data/region_latest_cl.xlsx")
print("기존 데이터 업로드 완료")

if '부스터스' in base.columns:
    base = base.drop(columns=['부스터스']) # 엑셀에 남아있는 부스터스 열 완전 삭제
if '공구우먼' not in base.columns:
    base['공구우먼'] = 0 # 과거 데이터는 공구우먼 0으로 초기화

if '지역' in base.columns:
    # '강남'을 포함하되 '강남구'가 아닌 행(강남신, 강남압, 강남청 등)을 제외하고 필터링
    base = base[~((base['지역'].str.contains('강남', na=False)) & (~base['지역'].str.contains('강남구', na=False)))]
    print("기존 엑셀 내 강남 파생 오염 데이터 제거 완료")
    
### ==================================================================================================================
# base 코드 내용 수정

# 동구 중구 -> 제물포구
# 1. 일단 동구와 중구의 이름을 제물포구로 통일합니다.
base.loc[base['지역'].isin(['인천광역시 동구', '인천광역시 중구']), '지역'] = '인천광역시 제물포구'

# 2. 숫자형 데이터를 합쳐주기 위해 그룹화(groupby)를 진행합니다.
# (문자열 컬럼인 '주문번호'나 '셀러명'이 있다면 상황에 맞게 처리하거나 제외해야 합니다)
# 아래는 '지역'과 '날짜'를 기준으로 나머지 숫자 데이터를 모두 더하는 코드입니다.
base = base.groupby(['날짜', '지역'], as_index=False).sum()


# 서구 -> 서해구 검단구
# 기존에 서해구 검단구에 없는 날짜 -> 서구에 있는 물량 가져올 것임
# 근데 서구 물량의 66%는 서해구로 가져오고 34%는 검단구로 넣기.


cond_seogu = base['지역'] == '인천광역시 서구'
seogu_data = base[cond_seogu].copy()

# 2. 메인 데이터프레임(base)에서 '인천광역시 서구' 행 삭제
base = base[~cond_seogu]

# 3. 분할 비율을 적용할 '숫자형(물량 등)' 컬럼만 추출 (날짜, 지역 컬럼 제외)
num_cols = seogu_data.select_dtypes(include='number').columns

# 4. 서해구 데이터 생성 (서구 물량의 66%)
seohae_data = seogu_data.copy()
seohae_data['지역'] = '인천광역시 서해구'
for col in num_cols:
    # 0.66을 곱한 후 정수로 변환 (소수점 버림)
    seohae_data[col] = (seohae_data[col] * 0.66).astype(int)

# 5. 검단구 데이터 생성 (서구 물량의 34%)
geomdan_data = seogu_data.copy()
geomdan_data['지역'] = '인천광역시 검단구'
for col in num_cols:
    # 총합 오차를 방지하기 위해 전체 서구 물량에서 서해구로 들어간 물량을 뺀 나머지를 전부 넣음
    geomdan_data[col] = seogu_data[col] - seohae_data[col]

# 6. 새로 만든 서해구, 검단구 데이터를 메인 데이터프레임(base)에 이어 붙이기
base = pd.concat([base, seohae_data, geomdan_data], ignore_index=True)

# 7. 기존에 서해구/검단구 데이터가 있었던 날짜가 있다면 하나로 합치기
base = base.groupby(['날짜', '지역'], as_index=False).sum()

### ================================================================================================================

# '강남신', '강남압', '강남청' 등의 텍스트가 포함된 모든 행을 '서울특별시 강남구'로 일괄 변경
df_forecast.loc[df_forecast['지역'].str.contains('강남', na=False), "지역"] = "서울특별시 강남구"

# 최신 데이터 전처리
df_forecast.loc[df_forecast['지역']=="경기도 남양주", "지역"] = "경기도 남양주시"
df_forecast.loc[df_forecast['지역']=="경기도 성남시 수정가", "지역"] = "경기도 성남시 수정구"

# 1. [핵심] 눈에 보이지 않는 양옆 공백 제거 및 문자열 강제 변환
# (매핑이 더 정확하게 작동하도록 공백 제거를 매핑 바로 전으로 살짝 끌어올렸습니다)
df_forecast['날짜'] = df_forecast['날짜'].astype(str).str.strip()
df_forecast['지역'] = df_forecast['지역'].astype(str).str.strip()

# [추가된 부분] 과거 데이터(base)도 띄어쓰기 오류 방지
if '지역' in base.columns:
    base['지역'] = base['지역'].astype(str).str.strip()


### ==============================================================================================================================
# [신규] 행정동 기반 중분류 섹터명 변경 일괄 매핑 (260915 엑셀 반영)
# - 엑셀에 명시된 60여 개의 세부 섹터명(예: 검단마전, 서해가일, 천안쌍이 등)을 상위 지역구로 깔끔하게 병합합니다.
def map_new_region_names(region_str):
    if not isinstance(region_str, str): return region_str
    
    if '검단' in region_str: return '인천광역시 검단구'
    if '서해' in region_str: return '인천광역시 서해구'
    if '계양' in region_str: return '인천광역시 계양구'
    if '천안' in region_str: return '충청남도 천안시'
    if '아산' in region_str: return '충청남도 아산시'
    
    # '남동구'가 '동구'로 오인되어 제물포구로 합쳐지는 현상 완벽 방지
    if '중구' in region_str and '인천' in region_str: return '인천광역시 제물포구'
    if '동구' in region_str and '인천' in region_str and '남동구' not in region_str: return '인천광역시 제물포구'
    
    return region_str

# df_forecast의 '지역' 컬럼에 매핑 함수 일괄 적용
df_forecast['지역'] = df_forecast['지역'].apply(map_new_region_names)

# 과거 엑셀 데이터(base)도 새 이름으로 싹 통일해서 묶어주기
base['지역'] = base['지역'].apply(map_new_region_names)
base = base.groupby(['날짜', '지역'], as_index=False).sum(numeric_only=True)
### ==============================================================================================================================


### 해당 내용 추가 ==============================================================================================================================
# 날짜가 7/1, 7/2 이고 지역이 '인천광역시 서구시'인 애들은 drop
# 7/4, 7/6, 7/7, 7/11, 7/12일에 지역명이 '인천광역시 서구시'인 애들은 '인천광역시 검단구'로 지역값 수정하기

# 2. 날짜가 7/1, 7/2 이고 지역이 '인천광역시 서구시'인 데이터 삭제(drop)
drop_cond = (df_forecast['날짜'].isin(['2026-07-01', '2026-07-02'])) & (df_forecast['지역'] == '인천광역시 서구시')
df_forecast = df_forecast[~drop_cond]

# 3. 7/4, 7/6, 7/7, 7/11, 7/12일에 지역명이 '인천광역시 서구시'인 데이터를 '인천광역시 검단구'로 수정
target_dates = ['2026-07-04', '2026-07-06', '2026-07-07', '2026-07-11', '2026-07-12']
modify_cond = (df_forecast['날짜'].isin(target_dates)) & (df_forecast['지역'] == '인천광역시 서구시')
df_forecast.loc[modify_cond, '지역'] = '인천광역시 검단구'

# 4. 검증 (아래 결과가 Empty DataFrame으로 나와야 정상적으로 삭제된 것입니다)
check_deleted = df_forecast[(df_forecast['날짜'].isin(['2026-07-01', '2026-07-02'])) & (df_forecast['지역'] == '인천광역시 서구시')]
print("삭제 후 남은 데이터 수:", len(check_deleted))
# =======================================================================================================================================


total = df_forecast.groupby(['날짜','지역'])['주문번호'].agg({('총 물량','count')}).reset_index()
zigzag = df_forecast[df_forecast['셀러명']=="지그재그 당일배송"].groupby(['날짜','지역'])['주문번호'].agg({('지그재그 당일배송','count')}).reset_index()
kakao = df_forecast[df_forecast['셀러명']=="카카오모빌리티"].groupby(['날짜','지역'])['주문번호'].agg({('카카오모빌리티','count')}).reset_index()
gonggu = df_forecast[df_forecast['셀러명']=="공구우먼"].groupby(['날짜','지역'])['주문번호'].agg({('공구우먼','count')}).reset_index()

total = pd.merge(total, zigzag, how = 'left', on = ['날짜','지역'])
total = pd.merge(total, kakao, how = 'left', on = ['날짜','지역'])
total = pd.merge(total, gonggu, how = 'left', on = ['날짜','지역']) # 부스터스 대신 공구우먼 병합

merge_data = total.groupby(['날짜','지역'])[['총 물량','지그재그 당일배송','카카오모빌리티','공구우먼']].sum().reset_index()
merge_data['카카오모빌리티'] = merge_data['카카오모빌리티'].astype(int)
merge_data['공구우먼'] = merge_data['공구우먼'].astype(int)

merge_data = merge_data.sort_values(by=['지역', '날짜']).reset_index(drop=True)


# 데이터 업데이트
data = pd.concat([base, merge_data], axis=0).reset_index(drop=True)
data['날짜'] = pd.to_datetime(data['날짜'], format='%Y-%m-%d', errors='coerce')
print("데이터 업데이트 완료")

# 중복 제거 및 날짜 & 지역 정렬
data.loc[data['지역']=="인천광역시 계양시", "지역"] = "인천광역시 계양구"
data.loc[data['지역']=="인천광역시 서구시", "지역"] = "인천광역시 서구"
data = data.drop_duplicates(subset=['날짜','지역'], keep='last').reset_index(drop=True)
data = data.sort_values(by=['지역', '날짜']).reset_index(drop=True)

## other_seller 정의
data['other_seller'] = data['총 물량'] - (data['지그재그 당일배송'] + data['카카오모빌리티'] + data['공구우먼'])
print("데이터 중복 제거")

# 데이터 오류 정정
## 서울특별시 금천구 물량 조정
data.loc[(data['날짜']=="2025-11-14") & (data['지역']=="서울특별시 금천구"), "총 물량"] = 246
data.loc[(data['날짜']=="2025-11-14") & (data['지역']=="서울특별시 금천구"), "지그재그 당일배송"] = 200
data.loc[(data['날짜']=="2025-11-14") & (data['지역']=="서울특별시 금천구"), "카카오모빌리티"] = 37
data.loc[(data['날짜']=="2025-11-14") & (data['지역']=="서울특별시 금천구"), "부스터스"] = 5
data.loc[(data['날짜']=="2025-11-14") & (data['지역']=="서울특별시 금천구"), "other_seller"] = 4
data.loc[(data['날짜']=="2025-11-15") & (data['지역']=="서울특별시 금천구"), "총 물량"] = 139
data.loc[(data['날짜']=="2025-11-15") & (data['지역']=="서울특별시 금천구"), "지그재그 당일배송"] = 126
data.loc[(data['날짜']=="2025-11-15") & (data['지역']=="서울특별시 금천구"), "카카오모빌리티"] = 13
data.loc[(data['날짜']=="2025-11-15") & (data['지역']=="서울특별시 금천구"), "부스터스"] = 0
data.loc[(data['날짜']=="2025-11-15") & (data['지역']=="서울특별시 금천구"), "other_seller"] = 0

# data.loc[data['날짜']=="2025-12-05","총 물량"] = (data.loc[data['날짜']=="2025-12-05","총 물량"]*0.92).astype(int)

## 2023-01-19일자 데이터 삭제
data = data[data['날짜'] != "2023-01-19"].sort_values(by=['지역', '날짜']).reset_index(drop=True)

# 신규 데이터 저장
data.to_excel("./test_data/region_latest_cl.xlsx", index=False)
print("신규 데이터 저장 완료")

print("학습 데이터 구축 완료!")
print("-"*70)

# ----------------------------------------------------------------------------------------------------------
# 기온 데이터 API 호출 및 병합

import requests
import pandas as pd
import numpy as np
import urllib3
from tqdm import tqdm
from datetime import datetime, timedelta

# -----------------------
# 0. 공통 설정
# -----------------------
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

serviceKey = "ZjfdmTgpRQm33Zk4KZUJIQ"

# ===== 단기예보 =====
def fetch_short_term_forecast(base_date: str, base_time: str, region_point: dict):
    """단기예보 API 호출 → 날짜별 평균기온 DataFrame 반환"""
    records_all = []

    for region, (nx, ny) in tqdm(region_point.items(), desc="단기예보 수집"):
        url = (
            f"https://apihub.kma.go.kr/api/typ02/openApi/VilageFcstInfoService_2.0/getVilageFcst"
            f"?pageNo=1&numOfRows=10000&dataType=JSON"
            f"&base_date={base_date}&base_time={base_time}"
            f"&nx={nx}&ny={ny}&authKey={serviceKey}"
        )
        try:
            response = requests.get(url, verify=False, timeout=20)
            response.raise_for_status()
            json_response = response.json()
            items = json_response["response"]["body"]["items"]["item"]
        except Exception as e:
            print(f"⚠️ {region} 단기 데이터 요청 실패: {e}")
            continue

        informations = {}
        for it in items:
            cate = it["category"]
            fcstDate = it["fcstDate"]
            fcstTime = it["fcstTime"]
            fcstValue = it["fcstValue"]
            key = f"{fcstDate}_{fcstTime}"
            if key not in informations:
                informations[key] = {}
            informations[key][cate] = fcstValue

        region_records = []
        for key, vals in informations.items():
            if "TMP" in vals:
                date = key.split("_")[0]
                region_records.append({"date": date, "TMP": float(vals["TMP"])})

        if not region_records:
            continue

        df_region = (
            pd.DataFrame(region_records)
            .groupby("date", as_index=False)["TMP"]
            .mean()
            .rename(columns={"TMP": "avg_temp"})
        )
        df_region["지역"] = region
        records_all.append(df_region)

    if not records_all:
        return pd.DataFrame(columns=["날짜", "지역", "avg_temp"])

    df_all = pd.concat(records_all, ignore_index=True)
    df_all.rename(columns={"date": "날짜"}, inplace=True)

    # 날짜 형식 통일
    df_all["날짜"] = pd.to_datetime(df_all["날짜"], format="%Y%m%d").dt.strftime("%Y-%m-%d")
    return df_all

# ===== 중기예보 =====
def fetch_mid_term_forecast(base_date: str, region_code: dict):
    """중기예보 API 호출 → 날짜별 평균기온 DataFrame 반환"""
    records = []

    for region, regId in tqdm(region_code.items(), desc="중기예보 수집"):
        url = (
            f"https://apihub.kma.go.kr/api/typ02/openApi/MidFcstInfoService/getMidTa"
            f"?pageNo=1&numOfRows=10000&dataType=JSON"
            f"&regId={regId}&tmFc={base_date}0600&authKey={serviceKey}"
        )
        try:
            response = requests.get(url, timeout=20)
            response.raise_for_status()
            data = response.json()["response"]["body"]["items"]["item"][0]
        except Exception as e:
            print(f"⚠️ {region}({regId}) 중기 데이터 요청 실패: {e}")
            continue

        for day in range(4, 11):
            ta_min = data.get(f"taMin{day}")
            ta_max = data.get(f"taMax{day}")
            if ta_min is not None and ta_max is not None:
                avg_temp = (ta_min + ta_max) / 2
                record = {
                    "날짜": (datetime.now() + timedelta(days=day)).strftime("%Y-%m-%d"),
                    "지역": region,
                    "avg_temp": avg_temp,
                }
                records.append(record)

    return pd.DataFrame(records, columns=["날짜", "지역", "avg_temp"])

# ===== 1) 지역 정의 =====
region_point = {
    '서울특별시': ('61','126'),
    '인천광역시': ('54','125'),
    '경기도': ('60','121'),
    '대전광역시': ('68','100'),
    '충청남도': ('65','110')
}
region_code = {
    '서울특별시': '11B10101',
    '인천광역시': '11B20201',
    '경기도': '11B20601',
    '대전광역시': '11C20401',
    '충청남도': '11C20101'
}

# ===== 2) 단기/중기 데이터 호출 =====
short_df = fetch_short_term_forecast(base_date=datetime.now().strftime("%Y%m%d"), base_time="0200", region_point=region_point)
mid_df = fetch_mid_term_forecast(base_date=datetime.now().strftime("%Y%m%d"), region_code=region_code)

# ===== 3) 단기 + 중기 통합 및 중복 제거 =====
combined_df = pd.concat([short_df, mid_df], ignore_index=True)

# 날짜 형식 완전 통일
combined_df["날짜"] = pd.to_datetime(combined_df["날짜"]).dt.strftime("%Y-%m-%d")

# 날짜 + 지역 기준 중복 제거 (단기 데이터 우선)
final_df = combined_df.drop_duplicates(subset=["날짜", "지역"], keep="first").reset_index(drop=True)

# 지역 + 날짜 정렬
final_df = final_df.sort_values(by=['지역','날짜']).reset_index(drop=True)

# avg_temp 소수점 첫째 자리 유효
final_df['avg_temp'] = np.round(final_df['avg_temp'], 1)

# 컬럼 순서 정렬
final_df = final_df[['날짜','지역','avg_temp']]

# 기존 기온 데이터 업로드
base_temp = pd.read_excel("./test_data/recent_temp.xlsx")
print("기존 기온 데이터 업로드 완료")

# 기온 데이터 업데이트
temp_data = pd.concat([base_temp, final_df], axis=0).reset_index(drop=True)
temp_data['날짜'] = pd.to_datetime(temp_data['날짜'], format='%Y-%m-%d', errors='coerce')
print("기온 데이터 업데이트 완료")

# 중복 제거
temp_data = temp_data.drop_duplicates(subset=['날짜','지역'], keep="last").reset_index(drop=True)
print("기온 데이터 중복 제거")

# 신규 기온 데이터 저장
temp_data = temp_data.sort_values(by=['지역','날짜'])
temp_data.to_excel("./test_data/recent_temp.xlsx", index=False)
print("신규 기온 데이터 저장 완료")

display(temp_data.tail(7))

# 날짜 별 기온 데이터 merge
## data의 지역명에서 공백 전까지만 추출
data["지역_key"] = data["지역"].str.split().str[0]
temp_data["지역_key"] = temp_data["지역"].str.split().str[0]

## 병합 (지역_key + 날짜 기준)
data = pd.merge(
    data,
    temp_data[["지역_key", "날짜", "avg_temp"]],
    on=["지역_key", "날짜"],
    how="left"
)

## 컬럼 정리
data = data.drop(columns=["지역_key"])
data = data.sort_values(by=['지역', '날짜']).reset_index(drop=True)
print("기온 데이터 병합 완료!")

display(data)


AutoContainerGeneration version: 1
Environment: prod-daas-dw-cluster.cluster-cgrjlsurd8vz.ap-northeast-2.rds.amazonaws.com
SSH 터널 연결 성공
쿼리 실행 완료
최신 DF 생성 완료
기존 데이터 업로드 완료
기존 엑셀 내 강남 파생 오염 데이터 제거 완료
삭제 후 남은 데이터 수: 0
데이터 업데이트 완료
데이터 중복 제거
신규 데이터 저장 완료
학습 데이터 구축 완료!
----------------------------------------------------------------------


중기예보 수집: 100%|██████████| 5/5 [00:01<00:00,  4.16it/s]


기존 기온 데이터 업로드 완료
기온 데이터 업데이트 완료
기온 데이터 중복 제거
신규 기온 데이터 저장 완료


,날짜,지역,avg_temp
5083,2026-10-31,충청남도,10.0
5084,2026-11-01,충청남도,9.0
5085,2026-11-02,충청남도,10.0
5086,2026-11-03,충청남도,8.5
5087,2026-11-04,충청남도,8.5
5088,2026-11-05,충청남도,9.0
5089,2026-11-06,충청남도,10.5


기온 데이터 병합 완료!


,날짜,지역,총 물량,지그재그 당일배송,카카오모빌리티,공구우먼,other_seller,부스터스,avg_temp
0,2024-01-02,경기도 고양시 덕양구,48,33,0,0,15,NaN,1.9
1,2024-01-03,경기도 고양시 덕양구,48,42,0,0,6,NaN,0.5
2,2024-01-04,경기도 고양시 덕양구,61,52,0,0,9,NaN,-0.2
3,2024-01-05,경기도 고양시 덕양구,53,47,0,0,6,NaN,3.5
4,2024-01-06,경기도 고양시 덕양구,27,27,0,0,0,NaN,-0.4
...,...,...,...,...,...,...,...,...,...
62382,2026-10-23,충청북도 청주시 흥덕구,218,101,1,9,107,NaN,NaN
62383,2026-10-24,충청북도 청주시 흥덕구,150,58,0,4,88,NaN,NaN
62384,2026-10-25,충청북도 청주시 흥덕구,165,60,0,0,105,NaN,NaN
62385,2026-10-26,충청북도 청주시 흥덕구,247,84,1,8,154,NaN,NaN


In [45]:
### Feature & Modeling
import numpy as np
import pandas as pd
import holidays
from tqdm import tqdm
from sklearn.metrics import mean_absolute_percentage_error
from xgboost import XGBRegressor

# =====================================
# 기온 체감도(Temperature Shift) 함수
# =====================================
def add_temp_shift_features(df, date_col="날짜", temp_col="avg_temp"):
    df = df.copy().sort_values(date_col)

    temp_ma10 = df[temp_col].rolling(7, min_periods=1).min()  # 최근 7일 rolling min
    diff = temp_ma10 - df[temp_col]  # 현재 기온과의 차이
    df["temp_shift"] = diff.abs()  # 절댓값
    df["temp_shift_sign"] = np.sign(diff)  # 방향

    df[["temp_shift", "temp_shift_sign"]] = df[["temp_shift", "temp_shift_sign"]].fillna(0.0)
    return df


# =====================================
# (지역별) 타겟별 lag/rolling 피처 생성
# =====================================
def add_target_lag_rolling_features_by_group(
    df,
    target_col,
    group_col="지역",
    lags=(1, 7, 14, 21, 28),
    roll_windows=(7, 14, 21, 28),
):
    df = df.copy()

    if group_col not in df.columns:
        raise KeyError(f"'{group_col}' 컬럼이 없습니다. 현재 컬럼: {df.columns.tolist()}")
    if "날짜" not in df.columns:
        raise KeyError(f"'날짜' 컬럼이 없습니다. 현재 컬럼: {df.columns.tolist()}")

    df = df.sort_values([group_col, "날짜"]).reset_index(drop=True)

    # 미래처럼 target이 없으면 컬럼만 생성
    if target_col not in df.columns:
        for lag in lags:
            df[f"{target_col}_lag_{lag}"] = np.nan
        for w in roll_windows:
            df[f"{target_col}_roll_mean_{w}"] = np.nan
            df[f"{target_col}_roll_std_{w}"] = np.nan
        return df

    # lag
    for lag in lags:
        df[f"{target_col}_lag_{lag}"] = df.groupby(group_col)[target_col].shift(lag)

    # rolling (shifted)
    shifted = df.groupby(group_col)[target_col].shift(1)

    for w in roll_windows:
        df[f"{target_col}_roll_mean_{w}"] = (
            shifted.groupby(df[group_col])
            .rolling(w, min_periods=max(2, w // 4))
            .mean()
            .reset_index(level=0, drop=True)
        )
        df[f"{target_col}_roll_std_{w}"] = (
            shifted.groupby(df[group_col])
            .rolling(w, min_periods=max(2, w // 4))
            .std()
            .reset_index(level=0, drop=True)
        )

    return df


# =====================================
# 전처리 함수 (지역별)
# =====================================
def preprocess_features_region(
    df,
    group_col="지역",
    manual_total_spike_up_mild_dates=None,
    manual_total_spike_up_strong_dates=None,
    manual_total_spike_down_dates=None,
    manual_zig_spike_up_mild_dates=None,
    manual_zig_spike_up_strong_dates=None,
    manual_zig_spike_down_dates=None,
):
    df = df.copy()
    df["날짜"] = pd.to_datetime(df["날짜"]).dt.normalize()
    df = df.sort_values(["날짜"] if group_col not in df.columns else [group_col, "날짜"]).reset_index(drop=True)

    # ===============================
    # 1) 날짜 기본 피처
    # ===============================
    df["month"] = df["날짜"].dt.month
    df["dow"] = df["날짜"].dt.weekday
    df["weekend"] = (df["dow"].isin([5, 6])).astype(int)
    df["week_of_year"] = df["날짜"].apply(lambda x: x.isocalendar().week)

    df["month"] = pd.Categorical(df["month"], categories=list(range(1, 13)))
    df["dow"] = pd.Categorical(df["dow"], categories=list(range(0, 7)))

    month_dummies = pd.get_dummies(df["month"], prefix="month").astype(int)
    dow_dummies = pd.get_dummies(df["dow"], prefix="dow").astype(int)
    df = pd.concat([df, month_dummies, dow_dummies], axis=1)

    df["month"] = df["month"].astype(int)
    df["dow"] = df["dow"].astype(int)

    # event
    event_periods = [
        ("2025-03-24", "2025-04-08"),
        ("2025-06-01", "2025-07-01"),
        ("2025-09-22", "2025-10-05"),
        ("2025-10-21", "2025-10-21"),
        ("2025-11-17", "2025-12-02"),
        ("2025-12-15", "2025-12-29"),
        ("2026-03-23", "2026-04-06"),
        ("2026-05-18", "2026-05-31"), # 지그재그 11주년 행사 
        ("2026-06-15", "2026-06-29"), # 직잿팟
                    
    ]
    df["is_event"] = 0
    for start, end in event_periods:
        mask = (df["날짜"] >= pd.to_datetime(start)) & (df["날짜"] <= pd.to_datetime(end))
        df.loc[mask, "is_event"] = 1

    # ===============================
    # 2) 지역별 auto spike
    # ===============================
    auto_cols = [c for c in ["총 물량", "지그재그 당일배송"] if c in df.columns]

    for c in ["총 물량", "지그재그 당일배송"]:
        df[f"{c}_spike_up_mild"] = 0
        df[f"{c}_spike_up_strong"] = 0
        df[f"{c}_spike_down"] = 0

    if group_col in df.columns:
        group_keys = df[group_col].dropna().unique().tolist()
    else:
        group_keys = [None]

    for col in auto_cols:
        trend_col = f"{col}_trend"
        resid_col = f"{col}_resid_ratio"
        df[trend_col] = np.nan
        df[resid_col] = np.nan

        for g in group_keys:
            mask_g = np.ones(len(df), dtype=bool) if g is None else (df[group_col] == g)

            # 동일 요일 baseline
            for d in range(7):
                mask_d = mask_g & (df["dow"] == d)
                idx_d = df.index[mask_d]
                if len(idx_d) == 0:
                    continue

                s = df.loc[idx_d, col].sort_index()
                baseline = s.shift(1).rolling(window=3, min_periods=2).mean()
                resid = (s - baseline) / baseline

                df.loc[idx_d, trend_col] = baseline
                df.loc[idx_d, resid_col] = resid

            # 평일/주말 threshold
            for wk_flag in [0, 1]:
                mask_wk = mask_g & (df["weekend"] == wk_flag) & df[resid_col].notnull()
                if mask_wk.sum() == 0:
                    continue

                if wk_flag == 0:
                    upper1, upper2, lower1 = 0.10, 0.20, -0.15
                else:
                    upper1, upper2, lower1 = 0.20, 0.25, -0.10

                mask_up_mild = mask_wk & (df[resid_col] >= upper1) & (df[resid_col] < upper2)
                mask_up_strong = mask_wk & (df[resid_col] >= upper2)
                mask_down = mask_wk & (df[resid_col] <= lower1)

                df.loc[mask_up_mild, f"{col}_spike_up_mild"] = 1
                df.loc[mask_up_strong, f"{col}_spike_up_strong"] = 1
                df.loc[mask_down, f"{col}_spike_down"] = 1

    # ===============================
    # 3) manual spike (날짜 기준, 모든 지역 동일 적용)
    # ===============================
    manual_total_spike_up_mild_dates = manual_total_spike_up_mild_dates or []
    manual_total_spike_up_strong_dates = manual_total_spike_up_strong_dates or []
    manual_total_spike_down_dates = manual_total_spike_down_dates or []
    manual_zig_spike_up_mild_dates = manual_zig_spike_up_mild_dates or []
    manual_zig_spike_up_strong_dates = manual_zig_spike_up_strong_dates or []
    manual_zig_spike_down_dates = manual_zig_spike_down_dates or []

    def _apply_manual(spike_info, colname):
        if not spike_info:
            return
            
        # 1. 리스트로 들어오면 전체 지역에 일괄 적용 (기존 방식)
        if isinstance(spike_info, list):
            dts = pd.to_datetime(spike_info).normalize()
            df.loc[df["날짜"].isin(dts), colname] = 1
            
        # 2. 딕셔너리로 들어오면 해당 날짜의 특정 지역만 적용
        elif isinstance(spike_info, dict):
            for date_str, regions in spike_info.items():
                dt = pd.to_datetime(date_str).normalize()
                df.loc[(df["날짜"] == dt) & (df["지역"].isin(regions)), colname] = 1

    _apply_manual(manual_total_spike_up_mild_dates, "총 물량_spike_up_mild")
    _apply_manual(manual_total_spike_up_strong_dates, "총 물량_spike_up_strong")
    _apply_manual(manual_total_spike_down_dates, "총 물량_spike_down")

    _apply_manual(manual_zig_spike_up_mild_dates, "지그재그 당일배송_spike_up_mild")
    _apply_manual(manual_zig_spike_up_strong_dates, "지그재그 당일배송_spike_up_strong")
    _apply_manual(manual_zig_spike_down_dates, "지그재그 당일배송_spike_down")

    # ===============================
    # 4) holiday / pre_holiday_peak
    # ===============================
    years = sorted(df["날짜"].dt.year.unique().tolist())
    kr_holidays = holidays.KR(years=years)

    df["holiday"] = df["날짜"].isin(kr_holidays).astype(int)
    df.loc[df["weekend"] == 1, "holiday"] = 0

    pre_holiday_map = {}
    for hd in kr_holidays:
        found_days = []
        offset = 1
        while len(found_days) < 2 and offset <= 7:
            prev_day = hd - pd.Timedelta(days=offset)
            if prev_day.weekday() not in [5, 6]:
                found_days.append(prev_day)
            offset += 1

        if len(found_days) >= 1:
            pre_holiday_map[found_days[0]] = 1
        if len(found_days) >= 2:
            if (found_days[0] - found_days[1]).days == 1:
                if pre_holiday_map.get(found_days[1], 0) != 1:
                    pre_holiday_map[found_days[1]] = 2

    df["pre_holiday_peak"] = df["날짜"].map(pre_holiday_map).fillna(0).astype(int)
    df.loc[df["holiday"] == 1, "pre_holiday_peak"] = 0

    post_holiday_map = {}
    for hd in kr_holidays:
        offset = 1
        while offset <= 7:
            next_day = hd + pd.Timedelta(days=offset)

            if next_day.weekday() not in [5, 6]:
                post_holiday_map[next_day] = 1
                break

            offset += 1

    df["post_holiday_peak"] = df["날짜"].map(post_holiday_map).fillna(0).astype(int)
    df.loc[df["holiday"] == 1, "post_holiday_peak"] = 0

    # =========================================
    # 5) 미운영 일자 holiday 제거
    # =========================================
    exclude_dates = pd.to_datetime(
        ["2024-02-09", "2024-02-10", "2024-02-11",
         "2024-08-17",
         "2024-09-16", "2024-09-17", "2024-09-18",
         "2025-01-28", "2025-01-29", "2025-01-30",
         "2025-10-05", "2025-10-06", "2025-10-07", "2025-10-08", "2025-10-09",
         "2026-02-16", "2026-02-17", "2026-02-18"]
    ).normalize()
    df.loc[df["날짜"].isin(exclude_dates), "holiday"] = 0

    # ===============================
    # 6) temp shift
    # ===============================
    if "avg_temp" in df.columns:
        df = add_temp_shift_features(df, temp_col="avg_temp")
        df = df.drop(columns=["avg_temp"], errors="ignore")

    # ===============================
    # 7) lag/rolling (지역별)
    # ===============================
    df = add_target_lag_rolling_features_by_group(
        df,
        "총 물량",
        group_col=group_col,
        lags=(1, 7, 14, 21, 28),
        roll_windows=(7, 14, 21, 28),
    )
    df = add_target_lag_rolling_features_by_group(
        df,
        "지그재그 당일배송",
        group_col=group_col,
        lags=(1, 7, 14, 21, 28),
        roll_windows=(7, 14, 21, 28),
    )

    # ===============================
    # 8) intermediate drop
    # ===============================
    drop_cols = []
    for col in auto_cols:
        drop_cols += [f"{col}_trend", f"{col}_resid_ratio"]

    drop_targets = ["카카오모빌리티", "부스터스", "other_seller"]
    drop_cols += drop_targets

    df = df.drop(columns=drop_cols, errors="ignore")
    return df


# =====================================
# early stopping best_iteration 예측
# =====================================
def _xgb_predict_with_best_iteration(model: XGBRegressor, X: pd.DataFrame) -> np.ndarray:
    if hasattr(model, "best_iteration") and model.best_iteration is not None:
        try:
            return model.predict(X, iteration_range=(0, model.best_iteration + 1))
        except TypeError:
            return model.predict(X)
    return model.predict(X)


def _safe_best_iteration(xgb_model):
    if xgb_model is None:
        return None
    bi = getattr(xgb_model, "best_iteration", None)
    if bi is not None:
        return int(bi)
    try:
        booster = xgb_model.get_booster()
        bi2 = getattr(booster, "best_iteration", None)
        return int(bi2) if bi2 is not None else None
    except Exception:
        return None


def _ensure_columns(df: pd.DataFrame, cols: list, fill_value=0.0) -> pd.DataFrame:
    df = df.copy()
    for c in cols:
        if c not in df.columns:
            df[c] = fill_value
    return df


# =====================================
# Custom Loss Function
# =====================================
def weighted_mse(alpha=1):
    def weighted_mse_fixed(label, pred):
        residual = (label - pred).astype("float")
        grad = np.where(residual > 0, -2 * alpha * residual, -2 * residual)
        hess = np.where(residual > 0, 2 * alpha, 2.0)
        return grad, hess
    return weighted_mse_fixed


# =====================================
# 지역별 학습/검증/예측
# =====================================
def train_validate_forecast_region_xgb(
    train_proc_region: pd.DataFrame,
    forecast_total_df: pd.DataFrame,
    forecast_days=7,
    valid_days=7,
    raw_data: pd.DataFrame = None,
    temp_future_df: pd.DataFrame = None,
    group_col="지역",
    future_manual_total_spike_up_mild_dates=None,
    future_manual_total_spike_up_strong_dates=None,
    future_manual_total_spike_down_dates=None,
    future_manual_zig_spike_up_mild_dates=None,
    future_manual_zig_spike_up_strong_dates=None,
    future_manual_zig_spike_down_dates=None,
    full_extra_trees=10,
):
    TOTAL = "총 물량"
    ZIG = "지그재그 당일배송"
    RANDOM_STATE = 42
    MIN_TRAIN_SAMPLES = 50

    if raw_data is None:
        raise ValueError("⚠️ raw_data(날짜+avg_temp 포함)를 넘겨줘야 future temp 생성이 가능합니다.")

    # -------------------------
    # 전체물동량(날짜별) 피처 구성
    # -------------------------
    total_item = train_proc_region.groupby("날짜")[TOTAL].sum().reset_index()
    total_item.rename(columns={TOTAL: "전체물동량"}, inplace=True)

    ft = forecast_total_df.copy()
    ft["날짜"] = pd.to_datetime(ft["날짜"]).dt.normalize()
    ft = ft.rename(columns={"pred_총물량": "전체물동량"})

    total_item["날짜"] = pd.to_datetime(total_item["날짜"]).dt.normalize()
    total_item = pd.concat([total_item, ft[["날짜", "전체물동량"]]], ignore_index=True)
    total_item = total_item.drop_duplicates(subset=["날짜"], keep="last").sort_values("날짜").reset_index(drop=True)

    # train merge
    train_proc_region = train_proc_region.copy()
    train_proc_region["날짜"] = pd.to_datetime(train_proc_region["날짜"]).dt.normalize()
    train_proc_region = pd.merge(train_proc_region, total_item, how="left", on="날짜")

    # temp source (global)
    temp_source = raw_data[["날짜", "avg_temp"]].copy()
    temp_source["날짜"] = pd.to_datetime(temp_source["날짜"]).dt.normalize()

    if temp_future_df is not None:
        tfd = temp_future_df.copy()
        tfd["날짜"] = pd.to_datetime(tfd["날짜"]).dt.normalize()
        tfd = tfd[["날짜", "avg_temp"]].copy()
    else:
        tfd = None

    # -------------------------
    # fallback helper
    # -------------------------
    def _make_short_history_fallback_forecast(df_region, forecast_total_df, group_col="지역"):
        """
        신규/짧은 이력 지역 fallback:
        마지막 실제값 기준 전체 대비 점유율로 미래 7일을 배분
        """
        region_name = df_region[group_col].iloc[0]
        hist = df_region.sort_values("날짜").copy()
        last_row = hist.iloc[-1]
        last_date = pd.to_datetime(last_row["날짜"]).normalize()

        total_on_last_date = train_proc_region.loc[
            train_proc_region["날짜"] == last_date, TOTAL
        ].sum()

        zig_on_last_date = train_proc_region.loc[
            train_proc_region["날짜"] == last_date, ZIG
        ].sum()

        total_share = float(last_row[TOTAL]) / float(total_on_last_date) if total_on_last_date > 0 else 0.0
        zig_share = float(last_row[ZIG]) / float(zig_on_last_date) if zig_on_last_date > 0 else 0.0

        future_fallback = forecast_total_df.copy()
        future_fallback["날짜"] = pd.to_datetime(future_fallback["날짜"]).dt.normalize()
        future_fallback[group_col] = region_name

        future_fallback["pred_총물량"] = np.round(
            future_fallback["pred_총물량"] * total_share
        ).clip(lower=0)

        if "pred_지그재그당일배송" in future_fallback.columns:
            future_fallback["pred_지그재그당일배송"] = np.round(
                future_fallback["pred_지그재그당일배송"] * zig_share
            ).clip(lower=0)
        else:
            future_fallback["pred_지그재그당일배송"] = 0.0

        return future_fallback[["날짜", group_col, "pred_총물량", "pred_지그재그당일배송"]]

    # -------------------------
    # 지역 loop
    # -------------------------
    results_valid = []
    results_forecast = []

    regions = train_proc_region[group_col].dropna().unique().tolist()

    global_last_date = train_proc_region["날짜"].max()

    for region in tqdm(regions, desc="지역 처리", unit="지역"):
        df_tr = (
            train_proc_region[train_proc_region[group_col] == region]
            .sort_values("날짜")
            .reset_index(drop=True)
            .copy()
        )

        # 신규/짧은 이력 지역 fallback 처리
        if len(df_tr) <= valid_days + 28:
            print(f"⚠️ {region}: 학습 이력이 부족하여 fallback 예측 사용")

            fallback_forecast = _make_short_history_fallback_forecast(
                df_region=df_tr,
                forecast_total_df=forecast_total_df,
                group_col=group_col,
            )
            results_forecast.append(fallback_forecast)
            continue

        split_idx = len(df_tr) - valid_days
        train_df = df_tr.iloc[:split_idx].copy()
        valid_df = df_tr.iloc[split_idx:].copy()

        last_date = global_last_date
        future_dates = pd.date_range(start=last_date + pd.Timedelta(days=1), periods=forecast_days)
        future_df = pd.DataFrame({"날짜": future_dates})
        future_df["날짜"] = pd.to_datetime(future_df["날짜"]).dt.normalize()
        future_df[group_col] = region

        # --- future avg_temp 구성 ---
        tail_temp = temp_source[temp_source["날짜"] <= last_date].tail(14)

        if tfd is not None:
            future_temp_part = pd.merge(future_df, tfd, on="날짜", how="left")
            last_temp = float(tail_temp["avg_temp"].iloc[-1]) if len(tail_temp) > 0 else 0.0
            future_temp_part["avg_temp"] = future_temp_part["avg_temp"].fillna(last_temp)
        else:
            last_temp = float(tail_temp["avg_temp"].iloc[-1]) if len(tail_temp) > 0 else 0.0
            future_temp_part = future_df.copy()
            future_temp_part["avg_temp"] = last_temp

        future_temp_input = pd.concat(
            [tail_temp.assign(**{group_col: region}), future_temp_part],
            ignore_index=True,
        )

        # --- future 전처리 ---
        future_feat_input = preprocess_features_region(
            future_temp_input,
            group_col=group_col,
            manual_total_spike_up_mild_dates=future_manual_total_spike_up_mild_dates,
            manual_total_spike_up_strong_dates=future_manual_total_spike_up_strong_dates,
            manual_total_spike_down_dates=future_manual_total_spike_down_dates,
            manual_zig_spike_up_mild_dates=future_manual_zig_spike_up_mild_dates,
            manual_zig_spike_up_strong_dates=future_manual_zig_spike_up_strong_dates,
            manual_zig_spike_down_dates=future_manual_zig_spike_down_dates,
        )
        future_feat_df = future_feat_input.iloc[-forecast_days:].reset_index(drop=True)

        # 미래에는 ZIG 컬럼 자체가 없을 수 있음 → chain용으로 미리 생성
        if ZIG not in future_feat_df.columns:
            future_feat_df[ZIG] = np.nan

        # 전체물동량 merge
        future_feat_df = pd.merge(future_feat_df, total_item, how="left", on="날짜")

        # -------------------------
        # 피처 풀 구성
        # -------------------------
        all_cols = df_tr.columns.tolist()

        spike_total_cols = [c for c in all_cols if c.startswith(f"{TOTAL}_spike_")]
        spike_zig_cols = [c for c in all_cols if c.startswith(f"{ZIG}_spike_")]

        total_ts_cols = [c for c in all_cols if c.startswith(f"{TOTAL}_lag_") or c.startswith(f"{TOTAL}_roll_")]
        zig_ts_cols = [c for c in all_cols if c.startswith(f"{ZIG}_lag_") or c.startswith(f"{ZIG}_roll_")]

        # (A) zig 모델: TOTAL 관련 spike/ts 제외, 날짜/지역/타겟 drop
        zig_base_drop = ["날짜", group_col, TOTAL, ZIG]
        zig_exclude = spike_total_cols + total_ts_cols
        zig_features = [c for c in all_cols if c not in zig_base_drop + zig_exclude]

        # (B) total 모델: ZIG는 feature로 사용(총 drop에서 제외), zig spike/ts 제외
        total_base_drop = ["날짜", group_col, TOTAL]
        total_exclude = spike_zig_cols + zig_ts_cols
        total_features = [c for c in all_cols if c not in total_base_drop + total_exclude]

        # -------------------------
        # (A) 지그재그 학습/검증 + FULL
        # -------------------------
        X_train_z = train_df[zig_features].fillna(0.0)
        y_train_z = train_df[ZIG].astype(float).values
        X_valid_z = valid_df[zig_features].fillna(0.0)
        y_valid_z = valid_df[ZIG].astype(float).values

        nz_mask_train_z = (y_train_z > 10)
        X_tr_nz_z = X_train_z.loc[nz_mask_train_z]
        y_tr_nz_z = np.log1p(y_train_z[nz_mask_train_z])

        if len(X_tr_nz_z) >= MIN_TRAIN_SAMPLES:
            xgb_val_z = XGBRegressor(
                n_estimators=5000,
                learning_rate=0.1,
                max_depth=7,
                subsample=0.7,
                colsample_bytree=0.7,
                reg_lambda=0.9,
                objective=weighted_mse(50),
                random_state=RANDOM_STATE,
                n_jobs=-1,
                early_stopping_rounds=100,
            )
            xgb_val_z.fit(
                X_tr_nz_z,
                y_tr_nz_z,
                eval_set=[(X_valid_z, np.log1p(np.clip(y_valid_z, 0, None)))],
                verbose=False,
            )
            pred_valid_z = np.expm1(_xgb_predict_with_best_iteration(xgb_val_z, X_valid_z))
            pred_valid_z = np.clip(pred_valid_z, 0, None)
        else:
            xgb_val_z = None
            mean_log = float(np.mean(y_tr_nz_z)) if len(y_tr_nz_z) > 0 else 0.0
            pred_valid_z = np.full(len(X_valid_z), np.expm1(mean_log))

        # MAPE zig
        maskz = (y_valid_z > 10)
        mape_zig = mean_absolute_percentage_error(y_valid_z[maskz], pred_valid_z[maskz]) if maskz.sum() > 0 \
            else mean_absolute_percentage_error(y_valid_z, pred_valid_z)

        # FULL zig
        best_it_z = _safe_best_iteration(xgb_val_z)
        full_n_estimators_z = 1500 if best_it_z is None else max(50, best_it_z + int(full_extra_trees))

        X_full_z = df_tr[zig_features].fillna(0.0)
        y_full_z = df_tr[ZIG].astype(float).values
        nz_full_z = (y_full_z > 10)
        X_full_nz_z = X_full_z.loc[nz_full_z]
        y_full_nz_z = np.log1p(y_full_z[nz_full_z])

        if len(X_full_nz_z) >= MIN_TRAIN_SAMPLES:
            xgb_full_z = XGBRegressor(
                n_estimators=full_n_estimators_z,
                learning_rate=0.1,
                max_depth=7,
                subsample=0.7,
                colsample_bytree=0.7,
                reg_lambda=0.9,
                objective=weighted_mse(50),
                random_state=RANDOM_STATE,
                n_jobs=-1,
            )
            xgb_full_z.fit(X_full_nz_z, y_full_nz_z, verbose=False)
        else:
            xgb_full_z = None

        # -------------------------
        # (B) 총물량 학습/검증 (검증은 chain: ZIG를 pred_valid_z로 주입)
        # -------------------------
        X_train_t = train_df[total_features].fillna(0.0)
        y_train_t = train_df[TOTAL].astype(float).values

        nz_mask_train_t = (y_train_t > 10)
        X_tr_nz_t = X_train_t.loc[nz_mask_train_t]
        y_tr_nz_t = np.log1p(y_train_t[nz_mask_train_t])

        X_valid_t_chain = valid_df[total_features].copy()
        if ZIG not in X_valid_t_chain.columns:
            X_valid_t_chain[ZIG] = 0.0
        X_valid_t_chain[ZIG] = pred_valid_z
        X_valid_t_chain = X_valid_t_chain.fillna(0.0)

        y_valid_t = valid_df[TOTAL].astype(float).values

        if len(X_tr_nz_t) >= MIN_TRAIN_SAMPLES:
            xgb_val_t = XGBRegressor(
                n_estimators=5000,
                learning_rate=0.1,
                max_depth=7,
                subsample=0.7,
                colsample_bytree=0.7,
                reg_lambda=0.9,
                objective=weighted_mse(50),
                random_state=RANDOM_STATE,
                n_jobs=-1,
                early_stopping_rounds=100,
            )
            xgb_val_t.fit(
                X_tr_nz_t,
                y_tr_nz_t,
                eval_set=[(X_valid_t_chain, np.log1p(np.clip(y_valid_t, 0, None)))],
                verbose=False,
            )
            pred_valid_t = np.expm1(_xgb_predict_with_best_iteration(xgb_val_t, X_valid_t_chain))
            pred_valid_t = np.clip(pred_valid_t, 0, None)
        else:
            xgb_val_t = None
            mean_log = float(np.mean(y_tr_nz_t)) if len(y_tr_nz_t) > 0 else 0.0
            pred_valid_t = np.full(len(X_valid_t_chain), np.expm1(mean_log))

        # MAPE total
        maskt = (y_valid_t > 10)
        mape_total = mean_absolute_percentage_error(y_valid_t[maskt], pred_valid_t[maskt]) if maskt.sum() > 0 \
            else mean_absolute_percentage_error(y_valid_t, pred_valid_t)

        # FULL total
        best_it_t = _safe_best_iteration(xgb_val_t)
        full_n_estimators_t = 1500 if best_it_t is None else max(50, best_it_t + int(full_extra_trees))

        X_full_t = df_tr[total_features].fillna(0.0)
        y_full_t = df_tr[TOTAL].astype(float).values
        nz_full_t = (y_full_t > 10)
        X_full_nz_t = X_full_t.loc[nz_full_t]
        y_full_nz_t = np.log1p(y_full_t[nz_full_t])

        if len(X_full_nz_t) >= MIN_TRAIN_SAMPLES:
            xgb_full_t = XGBRegressor(
                n_estimators=full_n_estimators_t,
                learning_rate=0.1,
                max_depth=7,
                subsample=0.7,
                colsample_bytree=0.7,
                reg_lambda=0.9,
                objective=weighted_mse(50),
                random_state=RANDOM_STATE,
                n_jobs=-1,
            )
            xgb_full_t.fit(X_full_nz_t, y_full_nz_t, verbose=False)
        else:
            xgb_full_t = None

        # -------------------------
        # 미래 예측: 재귀(ts overwrite) + chain (ZIG -> TOTAL)
        # -------------------------
        def _overwrite_ts_features(row_df: pd.DataFrame, tgt: str, buf: list):
            row = row_df.copy()
            for lag in (1, 7, 14, 21, 28):
                row[f"{tgt}_lag_{lag}"] = buf[-lag] if len(buf) >= lag else 0.0
            for w in (7, 14, 21, 28):
                tail = buf[-w:] if len(buf) >= w else buf
                if len(tail) >= 2:
                    row[f"{tgt}_roll_mean_{w}"] = float(np.mean(tail))
                    row[f"{tgt}_roll_std_{w}"] = float(np.std(tail, ddof=1))
                else:
                    row[f"{tgt}_roll_mean_{w}"] = 0.0
                    row[f"{tgt}_roll_std_{w}"] = 0.0
            return row

        zig_buf = df_tr[ZIG].astype(float).tolist()
        total_buf = df_tr[TOTAL].astype(float).tolist()

        # (1) 미래 ZIG 예측
        if xgb_full_z is None or len(zig_buf) == 0:
            fallback = float(np.mean(zig_buf[-7:])) if len(zig_buf) >= 7 else (float(np.mean(zig_buf)) if len(zig_buf) else 0.0)
            future_pred_zig = [fallback] * forecast_days
        else:
            preds = []
            for i in range(forecast_days):
                row = future_feat_df.iloc[[i]].copy()
                row = _overwrite_ts_features(row, ZIG, zig_buf)

                row = _ensure_columns(row, zig_features, fill_value=0.0)
                X_row = row[zig_features].fillna(0.0)

                pred_i = float(np.expm1(_xgb_predict_with_best_iteration(xgb_full_z, X_row))[0])
                pred_i = max(0.0, pred_i)

                preds.append(pred_i)
                zig_buf.append(pred_i)
            future_pred_zig = preds

        # (2) 미래 TOTAL 예측: ZIG를 예측치로 주입
        if xgb_full_t is None or len(total_buf) == 0:
            fallback = float(np.mean(total_buf[-7:])) if len(total_buf) >= 7 else (float(np.mean(total_buf)) if len(total_buf) else 0.0)
            future_pred_total = [fallback] * forecast_days
        else:
            preds = []
            for i in range(forecast_days):
                row = future_feat_df.iloc[[i]].copy()
                row[ZIG] = float(future_pred_zig[i])
                row = _overwrite_ts_features(row, TOTAL, total_buf)

                row = _ensure_columns(row, total_features, fill_value=0.0)
                X_row = row[total_features].fillna(0.0)

                pred_i = float(np.expm1(_xgb_predict_with_best_iteration(xgb_full_t, X_row))[0])
                pred_i = max(0.0, pred_i)

                preds.append(pred_i)
                total_buf.append(pred_i)
            future_pred_total = preds

        # -------------------------
        # 결과 저장
        # -------------------------
        valid_result = valid_df[["날짜"]].copy()
        valid_result[group_col] = region

        valid_result["true_총물량"] = y_valid_t
        valid_result["pred_총물량"] = pred_valid_t

        valid_result["true_지그재그당일배송"] = y_valid_z
        valid_result["pred_지그재그당일배송"] = pred_valid_z

        valid_result["APE_total"] = np.abs(valid_result["true_총물량"] - valid_result["pred_총물량"]) / np.maximum(
            1e-9, np.abs(valid_result["true_총물량"])
        )
        valid_result["APE_zigzag"] = np.abs(valid_result["true_지그재그당일배송"] - valid_result["pred_지그재그당일배송"]) / np.maximum(
            1e-9, np.abs(valid_result["true_지그재그당일배송"])
        )

        valid_result["MAPE_total_region"] = mape_total
        valid_result["MAPE_zigzag_region"] = mape_zig

        results_valid.append(valid_result)

        forecast_result = future_df[["날짜"]].copy()
        forecast_result[group_col] = region
        forecast_result["pred_총물량"] = future_pred_total
        forecast_result["pred_지그재그당일배송"] = future_pred_zig

        results_forecast.append(forecast_result)

    # -------------------------
    # Summary
    # -------------------------
    valid_all = pd.concat(results_valid, ignore_index=True) if len(results_valid) else pd.DataFrame()
    forecast_all = pd.concat(results_forecast, ignore_index=True) if len(results_forecast) else pd.DataFrame()

    if len(valid_all) > 0:
        mape_summary = (
            valid_all.groupby(group_col)[["APE_total", "APE_zigzag"]]
            .mean()
            .reset_index()
            .rename(columns={"APE_total": "MAPE_total", "APE_zigzag": "MAPE_zigzag"})
        )
    else:
        mape_summary = pd.DataFrame(columns=[group_col, "MAPE_total", "MAPE_zigzag"])

    return valid_all, forecast_all, mape_summary


In [46]:
### Prediction
print(data.columns.tolist())
print(data.head())

['날짜', '지역', '총 물량', '지그재그 당일배송', '카카오모빌리티', '공구우먼', 'other_seller', '부스터스', 'avg_temp']
          날짜           지역  총 물량  지그재그 당일배송  카카오모빌리티  공구우먼  other_seller  부스터스  \
0 2024-01-02  경기도 고양시 덕양구    48         33        0     0            15   NaN   
1 2024-01-03  경기도 고양시 덕양구    48         42        0     0             6   NaN   
2 2024-01-04  경기도 고양시 덕양구    61         52        0     0             9   NaN   
3 2024-01-05  경기도 고양시 덕양구    53         47        0     0             6   NaN   
4 2024-01-06  경기도 고양시 덕양구    27         27        0     0             0   NaN   

   avg_temp  
0       1.9  
1       0.5  
2      -0.2  
3       3.5  
4      -0.4  


In [47]:
# =========================
# 실행
# =========================
# 1) 지역 데이터 전처리
train_proc_region = preprocess_features_region(
    data,
    group_col="지역",
)
# 2) 지역별 학습/예측
valid_all, forecast_all, mape_summary = train_validate_forecast_region_xgb(
    train_proc_region=train_proc_region,
    
    # 🌟 수정: 이전 셀에서 만든 최종 결과물인 forecast_df_final 사용
    forecast_total_df=forecast_df_final.rename(columns={'총 물량': 'pred_총물량'}), 
    
    forecast_days=7,
    valid_days=7,
    raw_data=data,        
    temp_future_df=temp_merge_data,
    group_col="지역",
    
    future_manual_total_spike_down_dates={},
    future_manual_total_spike_up_mild_dates=[],
    future_manual_total_spike_up_strong_dates=[],
    future_manual_zig_spike_up_mild_dates=[],
    future_manual_zig_spike_up_strong_dates=[],
    future_manual_zig_spike_down_dates=[],
)

print("\n=== 지역별 Total MAPE 요약 ===")
display(mape_summary.sort_values("MAPE_total")[["지역", "MAPE_total"]].head())
display(mape_summary.sort_values("MAPE_total")[["지역", "MAPE_total"]].tail())

print("\n=== 지역별 zigzag MAPE 요약 ===")
display(mape_summary.sort_values("MAPE_zigzag")[["지역", "MAPE_zigzag"]].head())
display(mape_summary.sort_values("MAPE_zigzag")[["지역", "MAPE_zigzag"]].tail())

print("\n=== valid_all 샘플 ===")
display(valid_all.head())

print("\n=== forecast_all 샘플 ===")
display(forecast_all.head())


지역 처리: 100%|██████████| 75/75 [01:47<00:00,  1.44s/지역]


=== 지역별 Total MAPE 요약 ===


,지역,MAPE_total
22,경기도 안양시 만안구,0.024484
64,인천광역시 미추홀구,0.038373
44,서울특별시 노원구,0.039907
70,충청남도 천안시,0.040976
20,경기도 안산시 상록구,0.042898


,지역,MAPE_total
3,경기도 과천시,0.136733
23,경기도 양주시,0.137602
32,대전광역시 동구,0.137656
68,인천광역시 제물포구,0.163603
26,경기도 의왕시,0.340007



=== 지역별 zigzag MAPE 요약 ===


...

,지역,MAPE_zigzag
11,경기도 성남시 수정구,0.168349
18,경기도 시흥북,0.169758
32,대전광역시 동구,0.225387
23,경기도 양주시,0.887463
26,경기도 의왕시,1.360462



=== valid_all 샘플 ===


,날짜,지역,true_총물량,pred_총물량,true_지그재그당일배송,pred_지그재그당일배송,APE_total,APE_zigzag,MAPE_total_region,MAPE_zigzag_region
0,2026-10-21,경기도 고양시 덕양구,359.0,386.511078,146.0,148.135345,0.076633,0.014626,0.086794,0.115275
1,2026-10-22,경기도 고양시 덕양구,465.0,421.546082,207.0,160.383438,0.093449,0.225201,0.086794,0.115275
2,2026-10-23,경기도 고양시 덕양구,404.0,400.938202,194.0,194.640808,0.007579,0.003303,0.086794,0.115275
3,2026-10-24,경기도 고양시 덕양구,282.0,249.992859,125.0,119.840302,0.113501,0.041278,0.086794,0.115275
4,2026-10-25,경기도 고양시 덕양구,273.0,305.596191,97.0,117.020042,0.119400,0.206392,0.086794,0.115275



=== forecast_all 샘플 ===


,날짜,지역,pred_총물량,pred_지그재그당일배송
0,2026-10-28,경기도 고양시 덕양구,440.092621,142.772430
1,2026-10-29,경기도 고양시 덕양구,480.408813,188.213867
2,2026-10-30,경기도 고양시 덕양구,492.358826,199.648987
3,2026-10-31,경기도 고양시 덕양구,344.203583,131.651566
4,2026-11-01,경기도 고양시 덕양구,325.094788,107.821892


## Result Save

In [48]:
# ==========================================================
# [지역별 결과 저장] 
# (총 물량은 Cell 5에서 이미 저장했으므로 여기서는 지역 데이터만 저장합니다)
# ==========================================================

# 지역 정렬
forecast_all = forecast_all.sort_values(by=['지역','날짜'])

# 💡 현재 작업 중인 버전에 맞게 경로 지정 (v.6.4.1 또는 v.6.4.2)
VERSION_DIR = "./v.6.4.1"  # 👈 6.4.2 버전 파일에서는 "./v.6.4.2"로 수정해 주세요!
SAVE_DIR_REGION = f"{VERSION_DIR}/Region_results"

# 폴더가 없으면 자동 생성
# os.makedirs(SAVE_DIR_REGION, exist_ok=True)

# 파일명 설정 (원하시는 날짜명으로 변경하세요)
save_path_region = f"{SAVE_DIR_REGION}/Region_Forecasting_26_1028_1103.xlsx"

# 지역별 데이터만 단독 저장
forecast_all.to_excel(save_path_region, index=False)

print(f"🌍 지역별 예측 결과 저장 완료: {save_path_region}")

🌍 지역별 예측 결과 저장 완료: ./v.6.4.1/Region_results/Region_Forecasting_26_1028_1103.xlsx


In [52]:
import glob
import os
import pandas as pd

# ==========================================================
# [지역별 누적 검증 파일 생성]
# ==========================================================

# 💡 현재 작업 중인 버전에 맞게 경로 지정 (v.6.4.1 또는 v.6.4.2)
VERSION_DIR = "./v.6.4.1"  # 👈 6.4.2 버전 파일에서는 "./v.6.4.2"로 수정
SAVE_DIR_REGION = f"{VERSION_DIR}/Region_results"

# 검증 기간 (END_DATE = 실제 물량이 있는 마지막 날)
START_DATE = "2025-10-11"
END_DATE   = "2026-10-27"   # 👈 매주 갱신

# 자동 로딩에서 제외할 파일 (기간 겹침/임시 파일)
EXCLUDE_FILES = {
    "Region_Forecasting_26_0101_0106.xlsx",   # 25_1231_0106과 중복
    "Region_Forecasting_26_0827_0930.xlsx",   # 월 단위 파일 → 주간 예측 덮어씀
}

# 실제 물량 / 출력 경로
ACTUAL_PATH   = "./test_data/region_latest_cl.xlsx"   # 지역 로딩 셀이 갱신하는 최신 파일
FALLBACK_PATH = "./valid.xlsx"
OUTPUT_PATH   = f"{VERSION_DIR}/final_valid_merged.xlsx"

# -------------------------------------------------------------------------
# 1. 누적 예측 파일 자동 로딩 (파일명 순 = 시간 순)
# -------------------------------------------------------------------------
files = sorted(glob.glob(f"{SAVE_DIR_REGION}/Region_Forecasting_*.xlsx"))
files = [f for f in files if os.path.basename(f) not in EXCLUDE_FILES]

print(f"📂 {len(files)}개 파일 로딩: {os.path.basename(files[0])} ~ {os.path.basename(files[-1])}")
data = pd.concat([pd.read_excel(f) for f in files], axis=0)

# -------------------------------------------------------------------------
# 2. 데이터 전처리 및 정제
# -------------------------------------------------------------------------
data['날짜'] = pd.to_datetime(data['날짜'], errors='coerce')
data['지역'] = data['지역'].str.strip()

# 휴무일(기간) 0값 강제 할당
target_zero_cols = ["총 물량", "pred_총물량", "지그재그 당일배송", "pred_지그재그당일배송",
                    "카카오모빌리티", "부스터스", "other_seller"]
existing_cols = [c for c in target_zero_cols if c in data.columns]
holiday_mask = (data['날짜'] >= "2026-02-16") & (data['날짜'] <= "2026-02-18")
data.loc[holiday_mask, existing_cols] = 0

# 청주시(전체) vs 청주시 OO구(세분화) 중복 해결
has_district = data['지역'].str.contains('상당구|서원구|청원구|흥덕구')
is_city_total = data['지역'].str.contains('청주시') & ~has_district
dates_with_districts = data[has_district]['날짜'].unique()
data = data[~((data['날짜'].isin(dates_with_districts)) & is_city_total)]
data = data.drop_duplicates(subset=['날짜', '지역'], keep='last')

valid_region = data[
    (data['날짜'] >= pd.Timestamp(START_DATE)) &
    (data['날짜'] <= pd.Timestamp(END_DATE))
].sort_values(by=['지역', '날짜']).reset_index(drop=True)

if '예측 총 물량' in valid_region.columns:
    valid_region = valid_region.rename(columns={'예측 총 물량': 'pred_총물량'})

# -------------------------------------------------------------------------
# 3. 실제 물량 불러와서 Outer Join
# -------------------------------------------------------------------------
if os.path.exists(ACTUAL_PATH):
    valid = pd.read_excel(ACTUAL_PATH)
else:
    print(f"🚨 {ACTUAL_PATH} 파일이 없어 {FALLBACK_PATH}를 불러옵니다.")
    valid = pd.read_excel(FALLBACK_PATH)

valid['날짜'] = pd.to_datetime(valid['날짜'], errors='coerce')
valid['지역'] = valid['지역'].str.strip()

final_merged_data = pd.merge(
    valid,
    valid_region[['날짜', '지역', 'pred_총물량']],
    on=['날짜', '지역'],
    how='outer',
).sort_values(by=['지역', '날짜']).reset_index(drop=True)

# -------------------------------------------------------------------------
# 4. 복붙용 컬럼 정리 및 저장
# -------------------------------------------------------------------------
if '총 물량' not in final_merged_data.columns:
    final_merged_data['총 물량'] = None

final_merged_data['pred_총물량'] = final_merged_data['pred_총물량'].fillna(0)
final_merged_data['총 물량'] = final_merged_data['총 물량'].fillna(0)

temp_date = pd.to_datetime(final_merged_data['날짜'])
final_merged_data['필터용_날짜'] = temp_date.dt.date
final_merged_data['날짜'] = temp_date.dt.strftime('%Y. %m. %d')

final_merged_data = final_merged_data[['필터용_날짜', '날짜', '지역', 'pred_총물량', '총 물량']].copy()
final_merged_data.to_excel(OUTPUT_PATH, index=False)
print(f"✅ 최종 병합 파일 저장 완료: {OUTPUT_PATH}")

📂 56개 파일 로딩: Region_Forecasting_25_1010_1016.xlsx ~ Region_Forecasting_26_1028_1103.xlsx
✅ 최종 병합 파일 저장 완료: ./v.6.4.1/final_valid_merged.xlsx


In [34]:
import pandas as pd
import numpy as np

# 💡 현재 작업 중인 버전에 맞게 경로 지정 (v.6.4.1 또는 v.6.4.2)
VERSION_DIR = "./v.6.4.1"  # 👈 6.4.2 버전 파일에서는 "./v.6.4.2"로 수정해 주세요!
SAVE_DIR_REGION = f"{VERSION_DIR}/Region_results"
# 1. 파일 불러오기 (엑셀 파일 read_excel 사용)
input_path = f"{SAVE_DIR_REGION}/Region_Forecasting_26_1021_1028.xlsx" # 실제 가지고 계신 엑셀 파일명으로 맞춰주세요
df = pd.read_excel(input_path)


# 2. 컬럼명 정리
df.columns = df.columns.str.strip()

# 3. 날짜형 변환
df["날짜"] = pd.to_datetime(df["날짜"])

# 4. 날짜별 '최종 예측 물량(업데이트)' 맵핑 
# 해당 요일에 새로운 모델의 수치로 설정
target_map = {
pd.Timestamp("2026-10-26"): 33715,
pd.Timestamp("2026-10-27"): 30772,
}

# 5. 기존 날짜별 총물량 합계 구하기
df["기존_일일총합"] = df.groupby("날짜")["pred_총물량"].transform("sum")

# 6. 목표 일일총합 매핑 (업데이트 대상이 아닌 날짜는 기존 총합 유지)
df["목표_일일총합"] = df["날짜"].map(target_map).fillna(df["기존_일일총합"])

# 7. 배분 기준: 기존 총물량 비율
df["배분비율"] = np.where(
    df["기존_일일총합"] > 0,
    df["pred_총물량"] / df["기존_일일총합"],
    0
)

# 8. 총물량 적용 후 결과 산출 (소수점 둘째 자리 반올림)
df["총물량_적용후"] = (df["목표_일일총합"] * df["배분비율"]).round(2)

# 불필요한 계산용 컬럼 제거 및 정리
df_final = df[["날짜", "지역", "pred_총물량", "pred_지그재그당일배송", "총물량_적용후"]]

# 9. 결과 저장 (엑셀로 저장하기 위해 to_excel 사용)
output_path = "증가_지역별_예측값_총물량반영_1021_1028업데이트.xlsx"
df_final.to_excel(output_path, index=False)

print(f"저장 완료: {output_path}")

저장 완료: 증가_지역별_예측값_총물량반영_1021_1028업데이트.xlsx
